# RSNA Knee Abnormality Detection — 「RSNA Knee 0.937 | Weak-Label DINOv2 Meniscus Resid」解説付き写し

| | |
|---|---|
| **コンペ** | [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection) |
| **元notebook** | [RSNA Knee 0.937 \| Weak-Label DINOv2 Meniscus Resid](https://www.kaggle.com/code/renta0426/rsna-knee-0-937-weak-label-dinov2-meniscus-resid) |
| **原著者** | renta.k（`renta0426`） |
| **スコア** | Public LB **0.937**（V1 / 41 votes / 実行時間 2分51秒 · GPU T4 x2） |
| **日付** | 2026-09-05 |

> ⚠️ **これは学習目的の解説付き写しです。** コードは原著のものを一字一句そのまま保持し、出力と実行番号だけを消しています。実行はしていません。原著者への敬意として、必ず元notebookにupvoteしてください。
>
> このnotebookは**推論専用**です。モデルの重みはすべてKaggleデータセットとして添付されており、学習は別の場所で行われています。

---

## 手法の概要

膝MRIの1検査（study）から、**12個の所見**（ACL断裂、MCL断裂、内側・外側半月板損傷、内側・外側・膝蓋大腿の変形性関節症、関節液貯留、滑膜炎、ベーカー嚢腫、骨挫傷、骨折）を**同時に**確率で出すタスクです。

このnotebookの構造は非常に特徴的で、**「実績のある0.936の親notebookを一字一句そのまま実行し、12所見のうち1つだけを差し替える」**という設計になっています。

### 親（0.936）の4本の腕

| 腕 | 表現 | 役割 |
|---|---|---|
| DINOv2 frontier | 自己教師あり事前学習ViTの特徴量 | 汎用的な視覚特徴 |
| RadImageNet ResNet50 | 医用画像で事前学習したCNN | 医用ドメイン特化の特徴 |
| Raptor（CoAtNet）4視点 | Sagittal/Coronal/Axial × 2 の6スロット | 解剖学的な断面ごとの所見 |
| Transformer（88特徴のキャリブレーション） | 上記のメタ統合 | 所見別の較正 |

この4本をランク平均でブレンドしたものが Public LB 0.936（元notebook: [Head and shoulders, knees and toes](https://www.kaggle.com/code/renta0426/head-and-shoulders-knees-and-toes)）。

### 本notebookの唯一の変更点

**内側半月板（Medial Meniscus）だけ**、最終ランクを次の式で作り直す：

```
final_rank(Medial Meniscus) = 0.30 × Transformer rank
                            + 0.60 × Raptor rank
                            + 0.10 × fullfit0033 bag rank
```

そのうえで親と同じ平均タイ順位のパーセンタイル再ランクを掛ける。**残り11所見は親の提出物からバイト単位でそのままコピー**されます。

そしてそれを保証するために、親の提出ファイル・可視テストのUID一覧・親notebook本体の **SHA256ハッシュを事前にハードコードして照合**しています。「変えていない」という主張を、コードで証明しているわけです。

### 弱ラベル（weak label）という土台

セル7のドキュメントに、このコンペの本質的な難しさが書かれています：**4,407件のstudyのうち、構造化ラベルが付いているのはたった58件**。残りはすべて**自由記述の放射線科レポート**しかありません。

そこで採られたのが、レポートを言語モデルに読ませて12個の**確率**に変換する方法です。「断裂が疑われる（suspected）」というレポートは **1 ではなく 0.8** になります。曖昧な記述を無理やり0/1に潰さず、**曖昧さを曖昧さのまま目的変数にする**。これで学習可能なstudyが **58件 → 4,349件** に増えました。

---

## 評価指標

**指標**：**macro-averaged AUC**（12所見それぞれのROC-AUCを単純平均したもの）。

これは「**12個の完全に独立した二値分類問題を、平等に平均する**」という意味であり、そこから3つの帰結が直接導かれます：

1. **所見ごとに独立して最適化すべき。** 全所見に同じ後処理を掛ける理由はどこにもありません。ある所見だけ良くなる工夫があるなら、その所見にだけ適用すればよい。
2. **データ量に比例した労力配分は誤り。** 稀な所見（骨折）も頻繁な所見（変形性関節症）も、等しく **1/12** を占めます。症例数の多い所見に注力しても、点数は1/12しか動きません。
3. **順位不変なので、キャリブレーション単体では上がらない。** AUCは順位しか見ないので、確率の較正だけでは1点も動きません。

もし micro平均（全所見の予測を1つのプールにして計算）にしていたら、有病率の高い変形性関節症で総合点がほぼ決まり、**見逃すと重大な骨折の性能が埋もれてしまいます**。macro平均は「**どの所見も臨床的に等しく大事**」という価値判断を、そのまま数式にしたものです。

**このnotebookが指標をどう最適化しているか**：ここが本notebookの最大の教訓です。

macro平均は**所見ごとに分解できる**ので、「12所見のうち1つだけを改善する」施策が**そのまま総合点に 1/12 の重みで反映されます**。しかも他の11所見に一切影響しません。原著はこの構造を最大限に利用して、

- **内側半月板だけ**を狙い撃ちし（Raptorの重みを0.60まで上げ、専用のbagモデルを10%足す）、
- 残り11所見は**バイト単位で親をコピー**し、
- ハッシュ照合でそれを**検証可能な形で保証している**。

これは「指標の分解可能性を、実験設計そのものに変換した」例です。**変更を1所見に閉じ込めれば、その変更の効果は他の要因と混ざりません**——つまり測定できます。0.936 → 0.937 の +0.001 は、内側半月板1所見での改善が 1/12 に希釈された結果であり、逆算すれば**その所見単体では約 +0.012 の改善**があったことになります。


## セル1：キャッシュ捕捉シム（capture shim）

**何をしているか**：かなり変わったことをしています。`numpy.zeros` を**自作の関数で一時的に置き換え**（モンキーパッチ）、上流の `build_cache()` 関数が確保する配列を**横から観測**しています。

具体的には、`np.zeros` が呼ばれるたびに `inspect` で呼び出し元のフレームを覗き、「呼び出し元が `build_cache` で、形状が (studies数, 6スロット, 12スライス, 336, 336) なら、その配列への参照を記録する」ということをしています。

**なぜそうするのか**：後のセル（`fullfit0033` の推論）で、**同じDICOM画像をもう一度読み直したくない**からです。MRIのDICOM読み込みとリサンプリングは重い処理で、2回やれば実行時間が倍になります。

しかし親notebookのコードは「一字一句変更しない」という制約があるため、`build_cache()` に「作った配列を返す」機能を足すことができません。そこで**呼び出し元のコードを1文字も変えずに、確保された配列を外から捕まえる**という手段が採られました。

**なぜ変数名が全部 `_p33_` で始まるか**：親のコードと**名前空間が絶対に衝突しないようにする**ためです。同じnotebookの中で `import numpy as np` などをすると親の変数を壊す可能性があるので、すべてのローカル名にプレフィックスを付けています。

**評価**：技巧としては見事ですが、**これは正攻法ではありません**。`np.zeros` の差し替えは、上流が実装を少し変えただけ（例：`np.empty` に変更）で静かに機能しなくなります。「親を絶対に変えない」という制約が、コードをここまで複雑にしている——という**トレードオフの実例**として読むのが良いでしょう。

**用語補足**：*モンキーパッチ* = 実行時にライブラリの関数を差し替えること。強力ですが壊れやすいので、最後の手段です。

In [ ]:
"""Notebook cell source: capture the exact 336px DINO cache for public0033.

This file is embedded by ``build_public0033_meniscus10_notebook.py`` immediately
before the unchanged upstream DINO prediction cell.  The capture is deliberately
very narrow: it only observes the cache and mask allocation made by the upstream
``build_cache`` call for a 6-slot / 12-slice / 336px test volume.  It does not
modify the returned arrays or any parent prediction code.
"""

import builtins as _p33_builtins
import inspect as _p33_inspect
import numpy as _p33_np


_P33_CAPTURE_KEY = "_public0033_cache_capture_v1"

if hasattr(_p33_builtins, _P33_CAPTURE_KEY):
    raise RuntimeError("public0033: cache capture state already exists")

_p33_original_zeros = _p33_np.zeros


def _p33_normalize_shape(_p33_shape):
    try:
        return tuple(int(_p33_value) for _p33_value in _p33_shape)
    except TypeError:
        return None


def _p33_caller_studies():
    """Read only the local ``studies`` tuple from upstream ``build_cache``."""

    _p33_frame = _p33_inspect.currentframe()
    try:
        _p33_wrapper = None if _p33_frame is None else _p33_frame.f_back
        _p33_caller = None if _p33_wrapper is None else _p33_wrapper.f_back
        if _p33_caller is None or _p33_caller.f_code.co_name != "build_cache":
            return None
        _p33_value = _p33_caller.f_locals.get("studies")
        if not isinstance(_p33_value, (list, tuple)):
            return None
        _p33_studies = tuple(str(_p33_uid) for _p33_uid in _p33_value)
        if not _p33_studies or any(not _p33_uid for _p33_uid in _p33_studies):
            return None
        if len(set(_p33_studies)) != len(_p33_studies):
            raise RuntimeError("public0033: captured build_cache studies are not unique")
        return _p33_studies
    finally:
        # ``inspect`` frames retain all parent locals, including large DICOM objects.
        del _p33_frame
        del _p33_wrapper


_p33_state = {
    "schema_version": "public0033_cache_capture_v1",
    "original_zeros": _p33_original_zeros,
    "numpy_module": _p33_np,
    "events": [],
    "cache_candidates": [],
    "mask_candidates": [],
}


def _p33_zeros(
    _p33_shape,
    dtype=float,
    order="C",
    *,
    like=None,
    _p33_delegate=_p33_original_zeros,
    _p33_state_ref=_p33_state,
):
    """Delegate ``numpy.zeros`` while retaining only the prescribed allocations."""

    if like is None:
        _p33_value = _p33_delegate(_p33_shape, dtype=dtype, order=order)
    else:
        _p33_value = _p33_delegate(
            _p33_shape, dtype=dtype, order=order, like=like
        )

    _p33_shape_tuple = _p33_normalize_shape(_p33_shape)
    _p33_dtype = _p33_np.dtype(dtype)
    _p33_studies = _p33_caller_studies()
    _p33_event = {
        "shape": list(_p33_shape_tuple) if _p33_shape_tuple is not None else None,
        "dtype": _p33_dtype.str,
        "caller": "build_cache" if _p33_studies is not None else None,
        "study_count": len(_p33_studies) if _p33_studies is not None else None,
    }
    _p33_state_ref["events"].append(_p33_event)

    if _p33_studies is None:
        return _p33_value

    if (
        _p33_shape_tuple is not None
        and len(_p33_shape_tuple) == 5
        and _p33_shape_tuple[0] == len(_p33_studies)
        and _p33_shape_tuple[1:] == (6, 12, 336, 336)
        and _p33_dtype == _p33_np.dtype(_p33_np.uint8)
    ):
        _p33_state_ref["cache_candidates"].append(
            {"cache": _p33_value, "studies": _p33_studies}
        )
    elif (
        _p33_shape_tuple is not None
        and len(_p33_shape_tuple) == 2
        and _p33_shape_tuple[0] == len(_p33_studies)
        and _p33_shape_tuple[1:] == (6,)
        and _p33_dtype == _p33_np.dtype(_p33_np.float32)
    ):
        _p33_state_ref["mask_candidates"].append(
            {"mask": _p33_value, "studies": _p33_studies}
        )
    return _p33_value


_p33_np.zeros = _p33_zeros
setattr(_p33_builtins, _P33_CAPTURE_KEY, _p33_state)
del _p33_state
del _p33_original_zeros


## セル2：DINOv2 推論（親の第1の腕）

**何をしているか**：DICOMから膝MRIを再構成し、Meta の **DINOv2**（自己教師あり学習で訓練されたVision Transformer）を特徴抽出器として使い、12所見の確率を出します。ここが親パイプラインの最初の腕です。

主要な設定：

| 設定 | 値 | 意味 |
|---|---|---|
| `CROP_MM` | 130.0 | 膝を**物理サイズ130mm四方**で切り出す。ピクセル数ではなくmmで揃えるのがポイント |
| `CACHE_IMG` | 336 | 切り出した後の画素サイズ |
| `TARGETS` | 12所見 | ACL, MCL, 内側/外側半月板, 内側/外側/PF変形性関節症, 関節液, 滑膜炎, ベーカー嚢腫, 骨挫傷, 骨折 |
| `SEED` | 2026 | 再現性のための乱数固定 |
| `CACHE_BUDGET_GB` | 12.0 | 画像キャッシュのメモリ上限 |

**なぜそうするのか**：

- **mm単位での切り出しが決定的に重要です。** MRIは撮影装置や設定によって「1ピクセル何mm」が違います。ピクセル数で揃えると、装置が違うだけで膝の見かけの大きさが変わり、モデルは**病変ではなく撮影装置を学習してしまいます**。DICOMヘッダの `PixelSpacing` を使って物理サイズで揃えることで、この交絡を断ち切ります。医用画像の前処理では最も重要な作法の一つです。
- **DINOv2 を使う理由**：ラベルが極端に少ない（構造化ラベルは58件）ため、ImageNetの分類ラベルで事前学習したモデルよりも、**ラベル無しで大量の画像から学習した自己教師ありモデル**の方が汎用的な特徴を持っています。膝MRIはImageNetにほぼ存在しない画像ですが、DINOv2の特徴は「テクスチャや形状の一般的な表現」なので転移しやすい。
- `ThreadPoolExecutor` でDICOMヘッダ読み・ピクセル読み・スライス順序決定を**別々のスレッド数**（16 / 12 / 32）で並列化しています。I/O待ちが支配的な処理なので、スレッド並列が効きます。

**用語補足**：*DICOM* = 医用画像の標準フォーマット。画素だけでなく、撮影条件・患者情報・物理サイズなどのメタデータを含みます。*ViT (Vision Transformer)* = 画像をパッチに切って、自然言語処理のTransformerと同じ仕組みで処理するアーキテクチャ。

In [ ]:
from __future__ import annotations
import os
import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
_ASSET_ROOTS = [
    Path('/kaggle/input/rsna-knee-bend-dinov3-0917-repro-assets'),
    Path('/kaggle/input/datasets/tonylica/rsna-knee-bend-dinov3-0917-repro-assets'),
]
ASSET = next((path for path in _ASSET_ROOTS if (path / 'rsna-knee-weights' / 'manifest.json').is_file()),
             _ASSET_ROOTS[0])
_COMPETITION_ROOTS = [
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
]
ROOT = next((path for path in _COMPETITION_ROOTS if (path / 'train.csv').is_file()),
            _COMPETITION_ROOTS[0])
DINO = Path('/kaggle/input/models/metaresearch/dinov2/pytorch/small/1')
T0 = time.time()
DEVS = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
SEED = 2026
TARGETS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
CROP_MM = 130.0
CACHE_IMG = 336
GROUP = 3
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45
CACHE_BUDGET_MAX_GB = 24.0
CACHE_BUDGET_GB = 12.0
TEST_SHARE = 0.3
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
AUG_ROT_DEG = 8.0
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.1
LAT_MIN_OFFSET_MM = 20.0
SLICE_BAND = (0.2, 0.8)
RULES_NATIVE = {'order': 'normal', 'lat': 'centre', 'slot_fallback': False, 'decode_fill': 'nearest'}
RULES_LEGACY = {'order': 'dominant_axis', 'lat': 'corner_x', 'slot_fallback': True, 'decode_fill': 'zero'}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
SLOTS_RECOVERED = [('SAG_FLUID_FS', 'Sagittal', True, True), ('COR_FLUID_FS', 'Coronal', True, True), ('AX_FLUID_FS', 'Axial', True, True), ('SAG_FLUID_NOFS', 'Sagittal', True, False), ('COR_T1', 'Coronal', False, False), ('SAG_T1', 'Sagittal', False, False)]
SLOTS_PUBLIC = [('SAG_FLUID', 'Sagittal', None, True), ('COR_FLUID', 'Coronal', None, True), ('AX_FLUID', 'Axial', None, True), ('SAG_STRUCT', 'Sagittal', None, False), ('COR_STRUCT', 'Coronal', None, False), ('AX_STRUCT', 'Axial', None, False)]
SLOT_SCHEME = os.environ.get('SLOT_SCHEME', 'recovered')
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == 'public' else SLOTS_RECOVERED
N_SLOT = len(SLOTS)
POOL_PARTS = {'cls_mean': 2, 'cls_mean_focal': 3}
SLOT_PRIOR_TABLE = {'ACL': (0, 3, 5), 'MCL': (1, 4), 'Medial Meniscus': (0, 1, 3, 4), 'Lateral Meniscus': (0, 1, 3, 4), 'Medial OA': (1, 4, 5), 'Lateral OA': (1, 4, 5), 'PF OA': (0, 2, 5), 'Effusion': (0, 2), 'Synovitis': (0, 2), "Baker's": (0,), 'Contusion': (0, 1, 2), 'Fracture': (0, 1, 2, 4, 5)}
SLOT_PRIOR_STRENGTH = 0.55
FATSAT_OPTS = {'FS', 'FATSAT', 'FAT_SAT', 'FSAT'}
_SEP = re.compile('[_\\-.]')
_FATSAT_RX = re.compile('\\bfs\\b|fatsat|fat sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b')
_T1_RX = re.compile('\\bt1\\b|\\bt1w\\b')
_T2_RX = re.compile('\\bt2\\b|\\bt2w\\b')
_PD_RX = re.compile('\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens')

def log(msg):
    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)
IMG = CACHE_IMG

def available_gb():
    try:
        with open('/proc/meminfo') as fh:
            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}
        return int(info['MemAvailable'].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION

def plan_cache(n_study, n_test=0):
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))
    return groups
N_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))
CACHE_SLICES = GROUP * N_GROUP
HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']

def _hdr_vec(s, n):
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split('|')]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None

def side_from_geometry(h):
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)
        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)
        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))
        if ipp is None or iop is None or ps is None or (not rows) or (not cols):
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'
    return out

def side_from_corner_x(h):
    out = {}
    for st, g in h.groupby('StudyInstanceUID'):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'
    return out

def lat_of(h, tag=''):
    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)
    for st, g in h.groupby('StudyInstanceUID'):
        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]
        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:
            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]
        v = [x[0] for x in v if x and x[0] in ('L', 'R')]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')
    return d

def probe(item):
    split, study, series, path = item
    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}
    try:
        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))
        row['files'] = files
        row['n_slices'] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':
                row[t] = '|'.join((str(x) for x in v))
            else:
                row[t] = str(v)
    except Exception as exc:
        row['err'] = str(exc)[:120]
    return row

def walk(split):
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)

def annotate(df):
    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')
    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)
    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')
    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))
    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs
    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')
    te = pd.to_numeric(df['EchoTime'], errors='coerce')
    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')
    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))
    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))
    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])
    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')
    return df

def pick_slots(series_df, plane_map):
    series_df = series_df.copy()
    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)
    out = {}
    for study, g in series_df.groupby('StudyInstanceUID'):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g['plane'] == plane) & (g['fatsat'] == fs)
            if fluid is not None:
                sel &= g['fluid'] == fluid
            cand = g[sel]
            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):
                cand = g[(g['plane'] == plane) & ~g['fatsat']]
            if len(cand):
                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]
        out[study] = chosen
    return out
ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]
DECODE_FAILED = []

def _natural_key(name):
    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split('(\\d+)', str(name))))

def _order_dominant_axis(rec):
    files, d = (rec['files'], rec['dir'])
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=['ImagePositionPatient', 'InstanceNumber'])
            raw = getattr(ds, 'ImagePositionPatient', None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, 'InstanceNumber', None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))
    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float('inf'), r[3]))
    elif sum((r[2] is not None for r in rows)) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float('inf'), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return ([r[0] for r in rows], True)

def order_slices(rec):
    if RULES['order'] == 'dominant_axis':
        return _order_dominant_axis(rec)
    files, d = (rec['files'], rec['dir'])
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any((k is None for k, _ in keyed)):
        return (files, False)
    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)

def read_slot(rec, n_slice=None, out_size=None):
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = (rec.get('ordered') or rec['files'], rec['dir'], rec['px'])
    n = len(files)
    if n == 0:
        return None
    lo, hi = (int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1)))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])
    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, 'RescaleSlope', 1) or 1)
            ic = float(getattr(ds, 'RescaleIntercept', 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None
        planes.append(a)
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES['decode_fill'] == 'zero':
        if not got:
            DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get('SeriesInstanceUID', d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)
    if px and np.isfinite(px) and (px > 0):
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = (h // 2, w // 2)
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-06), 0, 1)
    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode='bilinear', align_corners=False)
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

def normalise_laterality(img, plane, lat):
    if lat != 'R':
        return img
    if plane in ('Coronal', 'Axial'):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])
ORDER_CACHE = os.environ.get('RSNA_ORDER_CACHE') or None

def build_cache(slot_map, plane_map, lat_map, tag):
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f'{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    n_job = len(jobs)
    t_ord = time.time()
    n_slice_total = sum((len(j[3]['files']) for j in jobs))
    log(f'{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)')
    ok = done = 0
    CHUNK_O = 1024
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec['SeriesInstanceUID'])
            if e and len(e['files']) == len(rec['files']):
                rec['ordered'] = e['files']
                ok += int(e['good'])
                hit += 1
        jobs = [j for j in jobs if 'ordered' not in j[3]]
        log(f'{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read')
    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(block, pool.map(lambda j: order_slices(j[3]), block)):
                rec['ordered'] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec['SeriesInstanceUID']] = {'files': files, 'good': bool(good)}
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f'{tag}: ordering budget spent at {done}/{len(jobs)}; the rest keep file order')
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix('.tmp')
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f'{tag}: ordered {ok}/{n_job} by geometry ({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s')
    jobs = [(st, k, plane, slot_map[st][name]) for st in studies for k, (name, plane, _, _) in enumerate(SLOTS) if name in slot_map[st]]
    log(f'{tag}: decoding {len(jobs)} slot-series')
    n_failed_before = len(DECODE_FAILED)
    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane, lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f'  {tag} {done}/{len(jobs)}')
            if time.time() - T0 > TIME_BUDGET:
                log(f'  {tag}: time budget reached during decode')
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f'{tag}: {int(mask.sum())}/{len(jobs)} slots filled' + (f'; {n_failed} series had a slice that would not decode' if n_failed else ''))
    gc.collect()
    return (studies, cache, mask)

class SlotHead(nn.Module):

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer('slot_prior', p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)
        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

class Model(nn.Module):

    def __init__(self, backbone, dim, pool='cls_mean', prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == 'cls_mean_focal':
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

def build_model(unfreeze_last, source=None, variant='small', pool='cls_mean', prior=False):
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError('DINOv2 weights not attached')
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))
    log(f'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}')
    return Model(bb, dim, pool=pool, prior=prior)
FINGERPRINT_TOL = 0.002

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out

def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=''):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to')
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(f'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.')
    log(f'{tag}fingerprint matches within {d:.2g}')
    return d

class WeightsError(RuntimeError):
    pass
TTA_OVERLAP = True
TTA_POOL = 'prob'
PUBLIC_FRONTIER_TARGET_POOL = {'Fracture': 'max', 'Contusion': 'max', 'Medial Meniscus': 'max', 'Lateral Meniscus': 'max', 'ACL': 'top2', 'MCL': 'top2', "Baker's": 'max'}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, 'Synovitis': 'original_mean'}
LEGACY_FOLD_SOFTPOOL_BETA = {'ACL': 6.0, 'MCL': 6.0, 'Medial Meniscus': 8.0, 'Lateral Meniscus': 8.0, "Baker's": 8.0, 'Contusion': 8.0, 'Fracture': 10.0}
LEGACY_FOLD_SOFTPOOL_ALPHA = {'ACL': 0.2, 'MCL': 0.2, 'Medial Meniscus': 0.25, 'Lateral Meniscus': 0.25, "Baker's": 0.2, 'Contusion': 0.2, 'Fracture': 0.15}

def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]

def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == 'max':
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == 'mean':
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == 'logit_mean':
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == 'original_mean':
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ('top2', 'top3'):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f'unknown TTA pooling mode for {target}: {mode}')
    return values

def legacy_fold_soft_window_pool(original_probs, target_idx):
    values = original_probs.mean(0).clone()
    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():
        j = target_idx[target]
        x = original_probs[:, :, j]
        weight = torch.softmax(float(beta) * x, dim=0)
        values[:, j] = (weight * x).sum(0)
    return values

@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError('predict_member was given no windows to average over')
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f'unknown target(s) in TTA_TARGET_POOL: {unknown}')
    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))
    model.eval()
    out, public_frontier_out, public_soft_out = ([], [], [])
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = ([], [], [])
        for st in starts:
            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = ([], [])
            for view in views:
                with torch.autocast('cuda', enabled=dev.type == 'cuda'):
                    z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            win_original_probs.append(view_probs[0])
        probs = torch.stack(win_probs)
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = torch.sigmoid(logits.mean(0)) if pool == 'logit' else probs.mean(0)
        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)
            public_frontier_out.append(public_v.cpu().numpy())
            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)
            public_soft_out.append(public_soft.cpu().numpy())
    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)
    if not return_public_frontier:
        return primary
    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)
    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)
    return (primary, public_frontier, public_soft)
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()

def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    t0 = time.time()
    with BUILD_LOCK:
        if 'state' in m:
            state, fp = (m['state'], None)
        else:
            ck = torch.load(Path(path) / m['file'], map_location='cpu', weights_only=False)
            state, fp = (ck['model'], ck.get('fingerprint'))
        model = build_model(int(m['config']['unfreeze_last']), variant=m['config']['variant'], pool=m['config'].get('pool', 'cls_mean'), prior=bool(m['config'].get('prior', False))).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no stored fingerprint (legacy bundle) -- accepted at reduced weight")
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m['id']).encode()).hexdigest()[:8], 16)
    public_member = 'state' not in m
    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)
    if public_member:
        p, public_p, public_soft = predicted
    else:
        p, public_p, public_soft = (predicted, None, None)
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == 'cuda':
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))

def _combine(per_member):
    all_ids = sorted({s for m in per_member for s in m['ids']})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get('target_weight')
        w = np.asarray(target_weight if target_weight is not None else [float(m.get('weight', 1.0))] * len(TARGETS), dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        r = pd.DataFrame(m['pred']).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m['ids']]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f'at least one target has no ensemble vote: {tot}')
    return (all_ids, acc / tot[None, :])

def combine_public_members_by_fold(per_member, pred_key='pred'):
    all_ids = sorted({study for member in per_member for study in member['ids']})
    position = {study: i for i, study in enumerate(all_ids)}
    groups = {}
    for i, member in enumerate(per_member):
        fold = member.get('fold')
        key = f'fold_{fold}' if fold is not None else f'member_{i}'
        groups.setdefault(key, []).append(member)
    fold_ranks, diagnostics = ([], [])
    for key, members_in_fold in sorted(groups.items()):
        matrices = []
        for member in members_in_fold:
            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)
            values[[position[study] for study in member['ids']]] = np.asarray(member[pred_key], np.float64)
            if np.isnan(values).any():
                raise WeightsError(f"{member.get('id')}: incomplete {pred_key} coverage")
            matrices.append(values)
        raw_fold_mean = np.mean(matrices, axis=0)
        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method='average', pct=True).to_numpy(np.float64))
        diagnostics.append({'ensemble_group': key, 'members': len(members_in_fold)})
    if len(fold_ranks) != 5:
        raise WeightsError(f'legacy branch requires five folds, found {len(fold_ranks)}')
    return (all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics))

def blend_legacy_frontier_and_soft(frontier_rank, soft_rank):
    output = np.asarray(frontier_rank, np.float64).copy()
    for j, target in enumerate(TARGETS):
        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))
        if alpha:
            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]
    return output

def infer_from_package(path, dev=None):
    man = json.loads((Path(path) / 'manifest.json').read_text())
    members = man['members']
    log(f'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)')
    test_df = pd.read_csv(ROOT / 'test.csv')
    test_series = pd.read_csv(ROOT / 'test_series.csv')
    plane_map = dict(zip(test_series['SeriesInstanceUID'], test_series['Anatomical_Plane']))
    hte = annotate(walk('test_series'))
    log(f'test header pass: {len(hte)} series')
    groups = {}
    for m in members:
        groups.setdefault(m['pixel_group'], []).append(m)
    groups.update(legacy_group_members())
    per_member, public_frontier_members = ([], [])
    est = {'fixed': None, 'win': None}

    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):
        if float(np.std(pred)) < 1e-09:
            log(f"  {m['id']}: degenerate predictions; not banked")
            return
        with STATE_LOCK:
            per_member.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': pred, 'weight': m.get('weight', 1.0), 'target_weight': m.get('target_weight'), 'holdout': m.get('holdout')})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-09:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append({'id': m['id'], 'fold': m.get('fold'), 'ids': ids, 'pred': public_pred, 'soft_pred': public_soft})
            elif public_pred is not None:
                log(f"  {m['id']}: public-frontier vote omitted because only {len(starts)} / {len(starts_full)} windows completed")
            all_ids, acc = _combine(per_member)
            write_submission(acc, all_ids, test_df, 'submission.csv')
            log(f"  banked {m['id']} fold {m.get('fold', '?')} ({len(starts)} window(s){(', jitter' if jitter else '')}); submission.csv = weighted rank mean of {len(per_member)} member(s)")
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, 'test '), f'test g{gi}')
        idx = np.arange(len(st_te))
        starts_full = window_starts(Cte.shape[2], GROUP)
        pending = sorted(gm, key=lambda m: -(m.get('holdout') or 0))
        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))

        def pop_next():
            with STATE_LOCK:
                if not pending:
                    return (None, None, False)
                left = TIME_BUDGET - (time.time() - T0)
                remaining = len(pending) + left_after
                slots_left = -(-remaining // len(DEVS))
                starts, jit = (starts_full, False)
                if est['fixed'] is not None and est['win'] is not None:
                    afford = max(left * 0.9, 0.0)
                    room = afford / max(slots_left, 1)
                    if est['fixed'] + est['win'] > room:
                        log(f'  {left / 60:.0f} min left: surrendering {len(pending)} member(s); not one more fits')
                        pending.clear()
                        return (None, None, False)
                    jit = est['fixed'] + 2 * len(starts_full) * est['win'] <= room * 0.6
                    per_win = est['win'] * (2 if jit else 1)
                    n_win = int((room - est['fixed']) / per_win) if per_win > 0 else len(starts_full)
                    n_win = max(1, min(len(starts_full), n_win))
                    if n_win < len(starts_full):
                        mid = (len(starts_full) - n_win) // 2
                        starts = starts_full[mid:mid + n_win]
                return (pending.pop(0), starts, jit)

        def worker(dev):
            others = [d for d in DEVS if d is not dev]
            while True:
                m, starts, jit = pop_next()
                if m is None:
                    return
                for attempt, d in enumerate([dev] + others[:1]):
                    try:
                        p, public_p, public_soft, (fs, ws) = _run_member(path, m, d, Cte, Mte, idx, starts, jit)
                        with STATE_LOCK:
                            est['fixed'], est['win'] = (fs, ws)
                        bank(m, st_te, p, starts, jit, public_p, public_soft)
                        break
                    except Exception as exc:
                        log(f"  MEMBER {m['id']} failed on {d} ({type(exc).__name__}: {exc}); " + ('retrying on peer device' if attempt == 0 and others else 'dropped -- costs one vote, not the run'))
                        if d.type == 'cuda':
                            with torch.cuda.device(d):
                                torch.cuda.empty_cache()
        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
        del Cte, Mte
        gc.collect()
    if not per_member:
        raise WeightsError('no member produced predictions; submission stays at 0.5')
    all_ids, acc = _combine(per_member)
    sub = write_submission(acc, all_ids, test_df, 'submission.csv')
    log(f'final submission.csv = weighted rank mean of {len(per_member)} member(s); {sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}')
    if len(public_frontier_members) == len(members):
        frontier_ids, frontier_acc = _combine(public_frontier_members)
        frontier_sub = write_submission(frontier_acc, frontier_ids, test_df, 'submission_public_0899.csv')
        log(f'submission_public_0899.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {frontier_sub.shape}; nulls {int(frontier_sub[TARGETS].isna().sum().sum())}')
        fold_ids, fold_frontier, fold_diagnostics = combine_public_members_by_fold(public_frontier_members, 'pred')
        soft_ids, fold_soft, _ = combine_public_members_by_fold(public_frontier_members, 'soft_pred')
        if fold_ids != soft_ids:
            raise WeightsError('legacy hard/soft study order mismatch')
        legacy_prediction = blend_legacy_frontier_and_soft(fold_frontier, fold_soft)
        legacy_sub = write_submission(legacy_prediction, fold_ids, test_df, 'submission_legacy_fold_blend.csv')
        fold_diagnostics.to_csv('legacy_fold_diagnostics.csv', index=False)
        log(f'legacy DINO aggregation written from five folds; {legacy_sub.shape}')
    else:
        log(f'public-frontier fallback not emitted: {len(public_frontier_members)} / {len(members)} required public members completed')
    return sub

def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg['img'])
    GROUP = int(cfg['group'])
    CACHE_SLICES = int(cfg['slices'])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg['crop_mm'])
    SLICE_BAND = tuple((float(x) for x in cfg['band']))
    rules = cfg.get('rules') or RULES_NATIVE
    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f'the members record pixel rules this pipeline cannot reproduce: {unknown}')
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg['slots']):
        raise WeightsError(f"the members were fitted on slots {cfg['slots']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")

def augment(imgs, generator=None):
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = (x.shape[0], x.device)
    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = (torch.cos(rot) / sc, torch.sin(rot) / sc)
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = (cos, -sin, tx)
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = (sin, cos, ty)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)

def write_submission(pred, studies, test_df, path):
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, 'StudyInstanceUID', studies)
    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub

def find_dinov2(variant='small'):
    if not (DINO / 'config.json').is_file():
        raise FileNotFoundError(DINO)
    return DINO

def legacy_group_members():
    return {}

def run_dinov2():
    path = ASSET / 'rsna-knee-weights'
    infer_from_package(path, DEVS[0])
    public = Path('/kaggle/working/submission_public_0899.csv')
    if not public.is_file():
        raise RuntimeError('public DINOv2 frontier was not produced')
    public.replace('/kaggle/working/submission.csv')
    for name in ('submission_legacy_fold_blend.csv', 'legacy_fold_diagnostics.csv'):
        candidate = Path('/kaggle/working') / name
        if candidate.is_file():
            candidate.unlink()
run_dinov2()


## セル3：fullfit0033 のキャッシュ済み推論の呼び出し

**何をしているか**：セル1で捕まえた画像キャッシュを1組だけ選び、**`np.zeros` を元に戻してから**、外部バンドル（データセットとして添付されたPythonモジュール）の `run_cached_inference(cache, mask, studies, output_paths)` を呼びます。

モジュールを読み込む前に **SHA256ハッシュを計算して照合**しています。

**なぜそうするのか**：

- **`np.zeros` を先に戻す**のは必須です。捕捉シムを付けたまま新しいコードを読み込むと、そのコードが確保する配列まで記録され続けてメモリを食い潰します。「フックは目的を達したら即座に外す」——これは副作用のあるパッチを書くときの鉄則です。
- **ハッシュ照合**は、実行するコードが**意図した版であることの証明**です。添付データセットが更新されて中身が変わっていたら、スコアの再現性は崩れます。ハッシュを固定しておけば、変わった瞬間に例外で止まります。
- **フォールバックを意図的に持たない**（docstringに "there is no prediction fallback" と明記）のも設計判断です。失敗したら黙って別の予測を使うのではなく、**落とす**。前日までのBiohubのnotebookと同じ「fail fast」思想です。

このセルの出力 `public0033_bag_raw.csv` が、後のセル9で内側半月板の10%成分として使われます。

**用語補足**：*SHA256* = ファイルの内容から計算される固定長の指紋。1バイトでも違えば全く違う値になるので、同一性の検証に使えます。

In [ ]:
"""Notebook cell source: invoke the bundled fullfit0033 cached inference runtime.

The preceding capture shim owns the only large array references.  This cell
selects exactly one cache/mask pair, restores ``numpy.zeros`` before loading
any new code, then calls the hash-bound runtime from the private bundle.
Failures are intentionally propagated: there is no prediction fallback.
"""

import builtins as _p33_builtins
import gc as _p33_gc
import hashlib as _p33_hashlib
import importlib.util as _p33_importlib_util
import json as _p33_json
import os as _p33_os
from pathlib import Path as _P33Path


_P33_CAPTURE_KEY = "_public0033_cache_capture_v1"
_P33_BUNDLE_SCHEMA = "public0033_meniscus10_bundle_v1"
_P33_RUNTIME_ENTRYPOINT = "run_cached_inference(cache, mask, studies, output_paths)"
_p33_work_dir = _P33Path(_p33_os.environ.get("PUBLIC0033_WORK_DIR", "/kaggle/working"))
_p33_output_paths = {
    "bag_raw_csv": str(_p33_work_dir / "public0033_bag_raw.csv"),
    "receipt_json": str(_p33_work_dir / "public0033_cached_inference_receipt.json"),
    "work_dir": str(_p33_work_dir),
}


def _p33_sha256_file(_p33_path):
    _p33_digest = _p33_hashlib.sha256()
    with _p33_path.open("rb") as _p33_handle:
        for _p33_block in iter(lambda: _p33_handle.read(8 << 20), b""):
            _p33_digest.update(_p33_block)
    return _p33_digest.hexdigest()


def _p33_restore_zeros(_p33_state):
    _p33_numpy = _p33_state.get("numpy_module")
    _p33_original = _p33_state.get("original_zeros")
    if _p33_numpy is None or _p33_original is None:
        raise RuntimeError("public0033: cache capture lacks original numpy.zeros")
    _p33_numpy.zeros = _p33_original


def _p33_capture_pair(_p33_state):
    _p33_caches = _p33_state.get("cache_candidates")
    _p33_masks = _p33_state.get("mask_candidates")
    if not isinstance(_p33_caches, list) or not isinstance(_p33_masks, list):
        raise RuntimeError("public0033: cache capture candidate registry is invalid")
    _p33_pairs = []
    for _p33_cache_record in _p33_caches:
        for _p33_mask_record in _p33_masks:
            _p33_studies = _p33_cache_record.get("studies")
            if _p33_studies != _p33_mask_record.get("studies"):
                continue
            _p33_cache = _p33_cache_record.get("cache")
            _p33_mask = _p33_mask_record.get("mask")
            if _p33_cache is None or _p33_mask is None:
                continue
            _p33_expected_n = len(_p33_studies)
            if (
                tuple(_p33_cache.shape) == (_p33_expected_n, 6, 12, 336, 336)
                and str(_p33_cache.dtype) == "uint8"
                and tuple(_p33_mask.shape) == (_p33_expected_n, 6)
                and str(_p33_mask.dtype) == "float32"
            ):
                _p33_pairs.append((_p33_cache, _p33_mask, _p33_studies))
    if len(_p33_pairs) != 1:
        raise RuntimeError(
            "public0033: expected exactly one captured 6x12x336 cache/mask pair, "
            f"got {len(_p33_pairs)}"
        )
    return _p33_pairs[0]


def _p33_find_bundle():
    _p33_input = _P33Path(_p33_os.environ.get("PUBLIC0033_INPUT_ROOT", "/kaggle/input"))
    if not _p33_input.is_dir():
        raise RuntimeError("public0033: /kaggle/input is unavailable")
    _p33_matches = []
    # Kaggle currently mounts datasets below
    # ``/kaggle/input/datasets/<owner>/<slug>``.  Retain the historical direct
    # mount form as well, but do not recurse through arbitrary dataset payloads.
    _p33_manifest_paths = set(_p33_input.glob("*/bundle_manifest.json"))
    _p33_manifest_paths.update(
        _p33_input.glob("datasets/*/*/bundle_manifest.json")
    )
    for _p33_manifest_path in sorted(
        _p33_manifest_paths, key=lambda _p33_path: _p33_path.as_posix()
    ):
        _p33_mount = _p33_manifest_path.parent
        _p33_manifest = _p33_json.loads(_p33_manifest_path.read_text(encoding="utf-8"))
        if _p33_manifest.get("schema_version") == _P33_BUNDLE_SCHEMA:
            _p33_matches.append((_p33_mount, _p33_manifest_path, _p33_manifest))
    if len(_p33_matches) != 1:
        raise RuntimeError(
            "public0033: expected exactly one private bundle manifest with schema "
            f"{_P33_BUNDLE_SCHEMA!r}, got {len(_p33_matches)}"
        )
    _p33_root, _p33_manifest_path, _p33_manifest = _p33_matches[0]
    _p33_runtime = _p33_manifest.get("runtime")
    if not isinstance(_p33_runtime, dict):
        raise RuntimeError("public0033: bundle runtime binding is missing")
    if _p33_runtime.get("bundle_relative_path") != "public0033_runtime.py":
        raise RuntimeError("public0033: bundle runtime path drift")
    if _p33_runtime.get("entrypoint") != _P33_RUNTIME_ENTRYPOINT:
        raise RuntimeError("public0033: bundle runtime entrypoint drift")
    _p33_runtime_sha = _p33_runtime.get("sha256")
    if not isinstance(_p33_runtime_sha, str) or len(_p33_runtime_sha) != 64:
        raise RuntimeError("public0033: bundle runtime SHA256 is invalid")
    _p33_runtime_path = _p33_root / "public0033_runtime.py"
    if not _p33_runtime_path.is_file() or _p33_sha256_file(_p33_runtime_path) != _p33_runtime_sha:
        raise RuntimeError("public0033: bundled runtime SHA256 mismatch")
    return _p33_root, _p33_manifest_path, _p33_manifest, _p33_runtime_path


_p33_state = getattr(_p33_builtins, _P33_CAPTURE_KEY, None)
if not isinstance(_p33_state, dict):
    raise RuntimeError("public0033: cache capture state is missing")

_p33_cache = None
_p33_mask = None
_p33_studies = None
try:
    _p33_cache, _p33_mask, _p33_studies = _p33_capture_pair(_p33_state)
    # The shim is only for the unchanged parent DINO cell.  The private runtime
    # must receive normal NumPy semantics, even if it allocates additional buffers.
    _p33_restore_zeros(_p33_state)
    _p33_root, _p33_manifest_path, _p33_manifest, _p33_runtime_path = _p33_find_bundle()
    _p33_spec = _p33_importlib_util.spec_from_file_location(
        "public0033_runtime", _p33_runtime_path
    )
    if _p33_spec is None or _p33_spec.loader is None:
        raise RuntimeError("public0033: cannot load bundled runtime")
    _p33_module = _p33_importlib_util.module_from_spec(_p33_spec)
    _p33_spec.loader.exec_module(_p33_module)
    _p33_runner = getattr(_p33_module, "run_cached_inference", None)
    if not callable(_p33_runner):
        raise RuntimeError("public0033: run_cached_inference is missing from bundle")
    _p33_result = _p33_runner(
        _p33_cache,
        _p33_mask,
        list(_p33_studies),
        dict(_p33_output_paths),
    )
    if not isinstance(_p33_result, dict) or _p33_result.get("status") != "passed":
        raise RuntimeError("public0033: cached inference runtime did not report status=passed")
    if _p33_result.get("bag_raw_csv") != _p33_output_paths["bag_raw_csv"]:
        raise RuntimeError("public0033: cached inference output path drift")
    if not _P33Path(_p33_output_paths["bag_raw_csv"]).is_file():
        raise RuntimeError("public0033: cached inference did not create bag_raw.csv")
finally:
    # Always undo the parent shim and drop its only strong references to the cache.
    _p33_restore_zeros(_p33_state)
    _p33_state.get("cache_candidates", []).clear()
    _p33_state.get("mask_candidates", []).clear()
    _p33_state.get("events", []).clear()
    if getattr(_p33_builtins, _P33_CAPTURE_KEY, None) is _p33_state:
        delattr(_p33_builtins, _P33_CAPTURE_KEY)
    _p33_cache = None
    _p33_mask = None
    _p33_studies = None
    _p33_gc.collect()
    try:
        import torch as _p33_torch

        if _p33_torch.cuda.is_available():
            _p33_torch.cuda.empty_cache()
    except Exception:
        # Cleanup must not hide the primary error; no prediction fallback is made.
        pass


## セル4：Raptor（CoAtNet）4視点確率アンサンブル

**何をしているか**：親の第2の腕。膝MRIを **6つのスロット** — `(Sagittal, 1), (Sagittal, 0), (Coronal, 1), (Coronal, 0), (Axial, 1), (Axial, 0)` — に分けて、それぞれから16スライスを抽出し、CoAtNetベースのモデルで推論します。最後に既存の submission と**ランクで重み付き平均**します。

主要な設定：

| 設定 | 値 | 意味 |
|---|---|---|
| `SLOTS` | Sagittal/Coronal/Axial × 2 | 矢状断・冠状断・横断の3方向 × 2系列 |
| `N_SLICE` | 16 | 各スロットから16枚のスライスを取る |
| `SLICE_BAND` | (0.12, 0.88) | **端12%を捨てて中央76%だけ使う**。端のスライスは膝から外れていることが多い |
| `SIZE` | 336 | 画素サイズ |
| `INTENSITY` | 'slice' | 輝度正規化をスライス単位で行う |
| `A5_W` | ブレンド重み | 既存submissionとのランク混合比 |

**なぜそうするのか**：

- **断面の向きごとに見える所見が違います。** 前十字靭帯（ACL）は矢状断（Sagittal）で最もよく見え、半月板は冠状断（Coronal）で内側・外側の区別がつきやすく、膝蓋大腿関節は横断（Axial）が要ります。**放射線科医が実際に3方向を見比べる**のと同じことを、モデルの入力設計に落とし込んでいます。臨床知識が直接アーキテクチャになっている良い例です。
- `SLICE_BAND = (0.12, 0.88)` は地味ですが効きます。MRIシリーズの最初と最後のスライスは、膝関節から外れた大腿骨や脛骨の途中であることが多く、ノイズにしかなりません。
- **ランクでブレンドしている**（`rank(method='average', pct=True)`）のが重要です。異なるモデルの出す確率は**スケールが揃っていません**（片方は0.1〜0.3、もう片方は0.4〜0.9に固まっている等）。そのまま平均すると、値域の広いモデルが不当に支配します。パーセンタイル順位に変換してから混ぜれば、**スケールの違いを完全に無視**できます。macro-AUCは順位しか見ないので、この変換で失うものは何もありません。
- 最後の `assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'` は、列の順序が崩れていないかの確認。列がずれたまま提出すると、**全所見の予測が入れ替わって壊滅的なスコア**になります。1行のassertで防げる事故です。

**用語補足**：*CoAtNet* = 畳み込み（CNN）とAttention（Transformer）を段階的に組み合わせたアーキテクチャ。浅い層でCNNの局所性、深い層でAttentionの大域性を使えます。

In [ ]:
_A5_SAVED = dict(globals())
import gc, os, time, warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings('ignore')
cv2.setNumThreads(1)
CROP_MM = 130.0
SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICE = 16
INTENSITY = 'slice'
SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1), ('Axial', 0)]
N_SLOT = len(SLOTS)
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_COMPETITION_ROOTS = [
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
]
COMP = next((path for path in _COMPETITION_ROOTS if (path / 'train.csv').is_file()),
            _COMPETITION_ROOTS[0])
CKPT = ASSET / 'knee-mri-fold-weights'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'competition : {COMP}')
print(f'checkpoints : {CKPT}')
print(f'device      : {DEV}')
for i in range(torch.cuda.device_count() if DEV == 'cuda' else 0):
    cc = torch.cuda.get_device_capability(i)
    print(f'  gpu{i}       : {torch.cuda.get_device_name(i)} sm_{cc[0]}{cc[1]}, {torch.cuda.get_device_properties(i).total_memory / 2 ** 30:.0f} GiB, native bf16={cc >= (8, 0)}')
SERIES_ROOT = COMP / 'test_series'
if not SERIES_ROOT.exists():
    SERIES_ROOT = COMP / 'train_series'
print('series root:', SERIES_ROOT)

def ordered_files(sdir, cap=64):
    keyed = []
    for f in sdir.glob('*.dcm'):
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            keyed.append((int(ds.InstanceNumber), str(f)))
        except Exception:
            continue
        if len(keyed) >= cap * 4:
            break
    return [f for _, f in sorted(keyed)]

def series_side(path):
    try:
        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])
    except Exception:
        return 0.0

def read_crop(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None
    try:
        ps = float(ds.PixelSpacing[0])
    except Exception:
        ps = CROP_MM / max(arr.shape)
    half = int(round(CROP_MM / ps / 2))
    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)
    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))
    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))
    crop = arr[y0:y1, x0:x1]
    return None if crop.size == 0 else crop

def window(crop, lo, hi, flip):
    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)
    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return img[:, ::-1].copy() if flip else img

def render(path, flip):
    crop = read_crop(path)
    if crop is None:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    return window(crop, lo, hi, flip)

def build_study(args):
    idx, study, recs = args
    out = np.zeros((N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
    mask = np.zeros(N_SLOT, np.uint8)
    rows = pd.DataFrame(recs)
    if len(rows):
        for s_i, (plane, fs) in enumerate(SLOTS):
            sub = rows[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
            if sub.empty:
                continue
            files = ordered_files(SERIES_ROOT / study / sub.iloc[0].SeriesInstanceUID)
            if not files:
                continue
            flip = plane != 'Sagittal' and series_side(files[0]) < 0
            lo, hi = SLICE_BAND
            i0 = int(round(lo * (len(files) - 1)))
            i1 = int(round(hi * (len(files) - 1)))
            avail = list(range(i0, i1 + 1))
            if len(avail) >= N_SLICE:
                picks = [avail[int(round(t))] for t in np.linspace(0, len(avail) - 1, N_SLICE)]
                off = 0
            else:
                picks, off = (avail, (N_SLICE - len(avail)) // 2)
            if INTENSITY == 'series':
                crops = [read_crop(files[p]) for p in picks]
                got = [x for x in crops if x is not None]
                if got:
                    samp = np.concatenate([x[::4, ::4].ravel() for x in got])
                    lo_, hi_ = np.percentile(samp, [1, 99])
                    for c, x in enumerate(crops):
                        if x is None:
                            x = read_crop(files[min(len(files) - 1, picks[c] + 1)])
                        if x is not None:
                            out[s_i, off + c] = (window(x, lo_, hi_, flip) * 255).astype(np.uint8)
            else:
                for c, p in enumerate(picks):
                    img = render(files[p], flip)
                    if img is None:
                        img = render(files[min(len(files) - 1, p + 1)], flip)
                    if img is not None:
                        out[s_i, off + c] = (img * 255).astype(np.uint8)
            mask[s_i] = len(picks)
    return (idx, out, mask)
sub_df = pd.read_csv(COMP / 'sample_submission.csv')
ser_csv = pd.read_csv(COMP / 'test_series.csv')
if not (COMP / 'test_series').exists():
    ser_csv = pd.read_csv(COMP / 'train_series.csv')
ser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]
studies = sub_df.StudyInstanceUID.tolist()
by = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}
print(f'{len(studies):,} test studies, {len(by):,} with series metadata')
N_SLOT_TYPES, MASK_IDX = (6, 0)

def segment_softmax(scores, sidx, B):
    T, K = scores.shape
    idx = sidx.unsqueeze(1).expand(-1, K)
    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)
    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)
    e = (scores - m[sidx]).exp()
    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)
    return e / s[sidx].clamp(min=1e-06)

class MeanMaxPool(nn.Module):

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        D = f.shape[1]
        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))
        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)
        mean = mean / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)
        return (torch.cat([mean, mx], 1), None)

class LabelAttentionPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):
        super().__init__()
        self.d, self.k, self.h = (d, n_labels, n_heads)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))
        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None

    def forward(self, f, sidx, B, slot=None, return_attn=False):
        scores = self.key(f) @ self.q.t() / self.d ** 0.5
        if self.slot_bias is not None and slot is not None:
            scores = scores + self.slot_bias.t()[slot]
        a = segment_softmax(scores, sidx, B)
        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)
        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))
        return (out, a)

class TokenXAttnPool(nn.Module):

    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):
        super().__init__()
        self.d, self.k = (d, n_labels)
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)

    def forward(self, tok, sidx, B, slot=None, return_attn=False):
        T, N, D = tok.shape
        cnt = torch.bincount(sidx, minlength=B)
        S = int(cnt.max().item())
        starts = torch.cumsum(cnt, 0) - cnt
        pos = torch.arange(T, device=tok.device) - starts[sidx]
        kv = tok + self.slot_emb(slot).unsqueeze(1)
        pad = tok.new_zeros(B, S, N, D)
        pad[sidx, pos] = kv
        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)
        keep[sidx, pos] = True
        kpm = ~keep.repeat_interleave(N, dim=1)
        pad = self.kv_norm(pad.reshape(B, S * N, D))
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        cls = tok[:, 0]
        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)
        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)
        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)
        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)
        return (torch.cat([att, base], -1), w)

class ViTSlotToken(nn.Module):

    def __init__(self, vit, n_cat, dim=None):
        super().__init__()
        self.vit = vit
        d = dim or vit.embed_dim
        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)
        self.num_features = vit.num_features
        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)
        vit.num_prefix_tokens = self._orig_prefix + 1
        for blk in vit.blocks:
            a = getattr(blk, 'attn', None)
            if a is not None and hasattr(a, 'num_prefix_tokens'):
                a.num_prefix_tokens = a.num_prefix_tokens + 1

    @staticmethod
    def _maybe(mod, x):
        return x if mod is None else mod(x)

    def forward_features(self, x, cat):
        v = self.vit
        x = v.patch_embed(x)
        pos = v._pos_embed(x)
        rope = None
        if isinstance(pos, tuple):
            x, rope = pos
        else:
            x = pos
        x = self._maybe(getattr(v, 'patch_drop', None), x)
        x = self._maybe(getattr(v, 'norm_pre', None), x)
        npt = self._orig_prefix
        tok = self.tok(cat).unsqueeze(1)
        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)
        if rope is not None:
            if getattr(v, 'rope_mixed', False):
                for i, blk in enumerate(v.blocks):
                    x = blk(x, rope=rope[i])
            else:
                for blk in v.blocks:
                    x = blk(x, rope=rope)
        else:
            x = v.blocks(x)
        return v.norm(x)

    def forward_head(self, x, pre_logits=True):
        return self.vit.forward_head(x, pre_logits=pre_logits)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class _GatedDepthBlock(nn.Module):

    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):
        super().__init__()
        self.norm = nn.GroupNorm(1, n_slice)
        self.v = nn.Conv2d(n_slice, n_slice, 1)
        self.g = nn.Conv2d(n_slice, n_slice, 1)
        self.out = nn.Conv2d(n_slice, n_slice, 1)
        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))
        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        z = self.norm(x)
        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))

class DepthCompress(nn.Module):

    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):
        super().__init__()
        self.imagenet = imagenet
        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])
        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)
        if imagenet:
            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))
            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))

    def forward(self, x):
        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)
        z = x
        for b in self.blocks:
            z = b(z)
        z = self.proj(z)
        if self.imagenet:
            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)
        return z * keep
N_PLANE, N_CONTRAST = (3, 2)
_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2
_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2

class SlotDepthMixer(nn.Module):

    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):
        super().__init__()
        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)
        self.alpha_max = alpha_max
        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])
        self.register_buffer('base', b.log()[self.r:])
        n_u = self.r + 1
        self.shared = nn.Parameter(torch.zeros(n_u))
        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))
        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))
        self.g0 = nn.Parameter(torch.zeros(()))
        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))
        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))
        idx = torch.arange(n_slice)
        self.register_buffer('off', idx[None, :] - idx[:, None])

    def kernel(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]
        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)
        return F.softmax(full, dim=-1)

    def alpha(self, slot):
        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))
        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])

    def forward(self, x, slot, vmask):
        T, S, H, W = x.shape
        if vmask is None:
            raise ValueError('stem=mixer requires the padding mask')
        k = self.kernel(slot)
        v = vmask.to(k.dtype)
        d = self.off + self.r
        inb = (d >= 0) & (d < self.ksize)
        kk = k[:, d.clamp(0, self.ksize - 1)] * inb
        M = kk * v[:, None, :]
        den = M.sum(-1, keepdim=True)
        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)
        ok = (den > 1e-06) & v[:, :, None].bool()
        M = torch.where(ok, M / den.clamp(min=1e-06), eye)
        a = self.alpha(slot)[:, None, None]
        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)
        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):
            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))
            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)
        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)

def _seg_mean_max(v, sidx, B):
    D = v.shape[1]
    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))
    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)
    mean = mean / cnt.clamp(min=1).unsqueeze(1)
    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)
    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)
    return torch.cat([mean, mx], 1)

def _pad_kv(x, sidx, B, norm):
    T, P, D = x.shape
    cnt = torch.bincount(sidx, minlength=B)
    S = int(cnt.max().item())
    starts = torch.cumsum(cnt, 0) - cnt
    pos = torch.arange(T, device=x.device) - starts[sidx]
    pad = x.new_zeros(B, S, P, D)
    pad[sidx, pos] = x
    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)
    keep[sidx, pos] = True
    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))

class _GatedDelta(nn.Module):

    def __init__(self, d, n_labels, n_heads, dropout):
        super().__init__()
        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.kv_norm = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.d_norm = nn.LayerNorm(d)
        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))
        self.db = nn.Parameter(torch.zeros(n_labels))
        self.gate = nn.Parameter(torch.zeros(n_labels))

    def delta(self, pat, sidx, B, return_attn):
        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)
        q = self.q.unsqueeze(0).expand(B, -1, -1)
        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)
        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)

class TokenResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class CodexResidualPool(_GatedDelta):

    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):
        super().__init__(d, n_labels, n_heads, dropout)
        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))
        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)
        return (base + self.gate * d_, w)

class ClsAddPool(nn.Module):

    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))

    def forward(self, tok, slot, sidx, B, pres, return_attn=False):
        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)

class Readout(nn.Module):

    def __init__(self, pool, d, n_labels=12, pe=64):
        super().__init__()
        self.pool_kind, self.k = (pool, n_labels)
        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)
        if pool in ('xres', 'clsadd', 'xcodex'):
            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)
        elif pool in ('attn', 'xattn'):
            if pool == 'xattn':
                self.pool = TokenXAttnPool(d, n_labels)
                wd = 3 * d + pe
            else:
                self.pool = LabelAttentionPool(d, n_labels)
                wd = d + pe
            self.norm = nn.LayerNorm(wd)
            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))
            self.b = nn.Parameter(torch.zeros(n_labels))
        else:
            self.pool = MeanMaxPool()
            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))
        self.drop = nn.Dropout(0.2)

    def forward(self, f, slot, sidx, B, return_attn=False):
        pe = self.pres_emb(slot)
        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)
        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):
            return self.pool(f, slot, sidx, B, pres)[0]
        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)
        if self.pool_kind in ('attn', 'xattn'):
            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)
            x = self.drop(self.norm(x))
            return (x * self.w).sum(-1) + self.b
        return self.net(torch.cat([pooled, pres], 1))

class Net(nn.Module):

    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):
        super().__init__()
        self.enc, self.cond = (enc, cond)
        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None
        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None
        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')
        D = enc.num_features
        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None
        self.readout = Readout(pool, D)
        if cond == 'post':
            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)

    def forward(self, im, slot, smeta, sidx, B, vm=None):
        if self.mixer is not None:
            im = self.mixer(im, slot, vm)
        if self.compress is not None:
            im = self.compress(im)
        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)
        if self.tokens:
            inner = getattr(self.enc, 'vit', self.enc)
            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))
            f = torch.cat([f[:, :1], f[:, orig:]], 1)
        else:
            f = self.enc.forward_head(f, pre_logits=True)
            if f.dim() > 2:
                f = f.flatten(1)
        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v
        if self.cond == 'post':
            f = f + ex(self.slot_emb(slot))
        if self.meta_mlp is not None and smeta.shape[1] > 0:
            mt = self.meta_mlp(smeta)
            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt
        return self.readout(f, slot, sidx, B)
models = []
for ckpt_path in sorted(CKPT.glob('*_f*.pt')):
    z = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    cfg = z['cfg']
    _stem = cfg.get('stem', 'native')
    _in = 3 if _stem == 'compress' else cfg.get('n_slice', 16)
    enc = timm.create_model(cfg['backbone'], pretrained=False, num_classes=0, in_chans=_in, **{'img_size': cfg['img']} if 'vit_' in cfg['backbone'] else {})
    if cfg['cond'] == 'token':
        enc = ViTSlotToken(enc, N_SLOT_TYPES)
    m = Net(enc, cfg['cond'], cfg.get('n_meta', 0), cfg['pool'], stem=_stem, n_slice=cfg.get('n_slice', 16))
    missing, unexpected = m.load_state_dict(z['state_dict'], strict=False)
    assert not missing, f'missing {missing[:5]}'
    assert not unexpected, f'unexpected {unexpected[:5]}'
    models.append(m.eval())
    print(f"loaded {ckpt_path.name}  fold {z['fold']}  {cfg['backbone']} pool={cfg['pool']} meta={cfg['meta']}")
CFG = cfg
assert CFG.get('n_meta', 0) == 0, f"checkpoint expects {CFG['n_meta']} metadata features -- build slot_meta for the TEST studies and pass it to predict() before submitting"
print(f"\n{len(models)} fold models ready | input norm: {CFG.get('norm', 'none')}")
AMP_PREF = 'bf16'

def amp_for(dev):
    if not str(dev).startswith('cuda'):
        return (torch.float32, False)
    cc = torch.cuda.get_device_capability(dev)
    if AMP_PREF == 'bf16':
        return (torch.bfloat16, True)
    if AMP_PREF == 'fp16':
        return (torch.float16, True)
    if AMP_PREF == 'fp32':
        return (torch.float32, False)
    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)
AMP_DT, AMP_ON = amp_for(DEV)
WORKERS = max(1, min(4, os.cpu_count() or 4))
CHUNK = 48
MICRO = 8
models = [m.to(DEV).eval() for m in models]
print(f"device {DEV} | amp {str(AMP_DT).split('.')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")

def _norm_(im):
    k = CFG.get('norm', 'none')
    if k == 'zscore':
        m = (im > 0).float()
        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)
        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n
        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n
        return (im - mu) / (var.sqrt() + 1e-06) * m
    if k == 'imagenet':
        m = (im > 0).float()
        return (im - 0.485) / 0.229 * m
    return im

@torch.no_grad()
def _micro(images, masks):
    dev = DEV
    ims, slots, sidx, vms = ([], [], [], [])
    for b in range(len(masks)):
        present = np.nonzero(masks[b] > 0)[0]
        if len(present) == 0:
            continue
        blk = images[b][present]
        ims.append(torch.from_numpy(blk))
        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))
        slots.append(torch.from_numpy(present + 1).long())
        sidx.append(torch.full((len(present),), b, dtype=torch.long))
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    if not ims:
        return out
    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))
    sl = torch.cat(slots).to(dev)
    si = torch.cat(sidx).to(dev)
    vm = torch.cat(vms).to(dev)
    sm = torch.zeros(len(sl), CFG.get('n_meta', 0), device=dev)
    per = torch.zeros(len(models), len(masks), len(LABELS), device=dev, dtype=torch.float32)
    with torch.autocast('cuda' if str(dev).startswith('cuda') else 'cpu', dtype=AMP_DT, enabled=AMP_ON):
        for fold_index, model in enumerate(models):
            per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(masks), vm=vm).float())
    got = per.cpu().numpy()
    keep = np.array([(masks[b] > 0).any() for b in range(len(masks))])
    out[:, keep] = got[:, keep]
    return out

def predict(images, masks):
    out = np.full((len(models), len(masks), len(LABELS)), np.nan, np.float32)
    for a in range(0, len(masks), MICRO):
        b = min(a + MICRO, len(masks))
        out[:, a:b] = _micro(images[a:b], masks[a:b])
    return out
preds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)
t0, done = (time.time(), 0)
with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    for c0 in range(0, len(studies), CHUNK):
        block = studies[c0:c0 + CHUNK]
        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)
        msks = np.zeros((len(block), N_SLOT), np.uint8)
        futs = [ex.submit(build_study, (i, s, by.get(s, []))) for i, s in enumerate(block)]
        for f in as_completed(futs):
            try:
                i, a, k = f.result()
                imgs[i], msks[i] = (a, k)
            except Exception as e:
                print(f'  study failed: {type(e).__name__}: {e}')
        preds[:, c0:c0 + len(block)] = predict(imgs, msks)
        done += len(block)
        el = time.time() - t0
        print(f'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m', flush=True)
        del imgs, msks
        gc.collect()
print(f'\ninference done in {(time.time() - t0) / 60:.1f} min')
A5_W = 0.45
A5_LABELS = list(LABELS)
_a5_ok = np.isfinite(preds).all(axis=(0, 2))
_a5_rank_mean = np.zeros((len(studies), len(LABELS)), np.float64)
for fold_index in range(preds.shape[0]):
    fold = preds[fold_index][_a5_ok]
    ordinal = fold.argsort(0).argsort(0).astype(np.float64)
    _a5_rank_mean[_a5_ok] += ordinal / max(len(fold) - 1, 1)
_a5_rank_mean /= preds.shape[0]
_a5_rank_mean[~_a5_ok] = np.nan
A5_PREDS = dict(zip(sub_df['StudyInstanceUID'].astype(str), _a5_rank_mean.astype(np.float32)))
for _a5k, _a5v in _A5_SAVED.items():
    globals()[_a5k] = _a5v
del _A5_SAVED, _a5k, _a5v
_a5_sub = pd.read_csv('/kaggle/working/submission.csv', dtype={'StudyInstanceUID': str})
assert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'
if A5_W > 0:
    _a5_ours = np.stack([A5_PREDS[_u] for _u in _a5_sub['StudyInstanceUID'].astype(str)])
    _a5_base_rank = _a5_sub[A5_LABELS].rank(method='average', pct=True)
    _a5_ours_rank = pd.DataFrame(_a5_ours, columns=A5_LABELS, index=_a5_sub.index).rank(method='average', pct=True)
    _a5_sub[A5_LABELS] = (1.0 - A5_W) * _a5_base_rank + A5_W * _a5_ours_rank
    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()
    _a5_sub.to_csv('/kaggle/working/submission.csv', index=False)


## セル5：RadImageNet ResNet50 による所見別キャリブレーション

**何をしているか**：親の第3の腕。**RadImageNet**（医用画像で事前学習されたモデル群）の ResNet50 を使い、その出力で既存の予測を較正します。

特徴的な設定：

```python
_RAD_ALPHA = 0.5                              # 混合の強さ
_RAD_EXCLUDE = ("Baker's", 'Fracture')        # この2所見には適用しない
_RAD_V48_SECOND_ALPHA = 0.15
```

そして重みファイルのSHA256を3つハードコードして照合しています。

**なぜそうするのか**：

- **`_RAD_EXCLUDE` が本notebookの思想を象徴しています。** ベーカー嚢腫と骨折の2つには、この較正を**適用しない**。macro平均AUCは12所見の独立な平均なので、「ある工夫が10所見では効くが2所見では悪化する」なら、**悪化する2所見だけ除外すれば、純粋に良いところだけ取れます**。micro平均だったらこの選択的適用は成立しません（全体が1つのプールなので分けられない）。**指標の構造が、実装の自由度を決めている**わけです。
- **RadImageNet を使う理由**：ImageNet（犬・猫・車）で事前学習したモデルより、CT/MRI/超音波の**医用画像で事前学習したモデル**の方が、膝MRIへの転移が効きます。DINOv2（汎用・自己教師あり）と RadImageNet（医用ドメイン特化）は**学習した特徴の質が違う**ので、アンサンブルの多様性源として理想的です。同じ事前学習のモデルを2つ混ぜても、誤りが相関していて効果は薄い。
- `try/except` で囲んで「較正に失敗したら生のTransformer出力をそのまま残す」フォールバックを持っています。セル3が「失敗したら落とす」だったのと対照的ですが、こちらは**あってもなくてもよい上乗せ**なので、失敗しても全体を止めない方が合理的です。**必須の処理は fail fast、任意の上乗せは safe fallback** ——使い分けが意識的です。

`_RAD_CAL_PAYLOAD` に base64+zlib で圧縮された較正パラメータが埋め込まれているのは、小さな数値テーブルをデータセットに分けずnotebook内に持つためのテクニックです。

**用語補足**：*キャリブレーション（較正）* = 予測確率を、実際の発生頻度に合うよう補正すること。AUCは順位不変なので**単調な較正は効きません**が、ここで行われているのは所見ごとに違う重みで別モデルと混ぜる操作なので**順位が変わり**、AUCに効きます。

In [ ]:
from __future__ import annotations
import contextlib as _rad_contextlib
import base64 as _rad_b64
import zlib as _rad_zlib
import gc as _rad_gc
import hashlib as _rad_hashlib
import json as _rad_json
import os as _rad_os
import re as _rad_re
import time as _rad_time
from concurrent.futures import ThreadPoolExecutor as _RadThreadPool
from pathlib import Path as _RadPath
import numpy as _rad_np
import pandas as _rad_pd
import pydicom as _rad_pydicom
import torch as _rad_torch
import torch.nn as _rad_nn
import torch.nn.functional as _rad_F
from torchvision.models import resnet50 as _rad_resnet50
_RAD_LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
_RAD_ALPHA = 0.5
_RAD_EXCLUDE = ("Baker's", 'Fracture')
_RAD_REFERENCE_HEADS_SHA256 = '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa'
_RAD_ENCODER_SHA256 = '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734'
_RAD_E13_HEADS_SHA256 = 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a'
_RAD_E13_MEMBER_WEIGHT = 0.5
_RAD_V48_SECOND_ALPHA = 0.15
_RAD_CAL_PAYLOAD = 'eNrtmk1vI8cRhv9KsJdcKKE/q6tzc4z4ZCMBcjQWhrCRDSG2ZEjaIEGQ/57n7RlRQ3KG4jqLJAcDS4o709NdXR9vvVU9/3z30+3N/bvffRuuawgxlm7eq2ePeffrpV8v/V9euvLraD3k6ilW6zn126vYd+U6eKmx9RiaF7OSx+X1weE67K7SdSgpVSauuaecUxr3rtp1LK0HyyV7y9Gmy/E6pBhTL61Zt2gWx+WtSew6JmOoM5AHutXp+ro8+ZrlsnvJOfDdfRL+Kl4zd2bqllNmga7rvrva2OzGNFuynGxpmnxrS/069hCtpt6T1dLLOQVsTbJ1fWunG2qXAX8NiU+7VK8rdqvNU89uwQwjpWyeLdTCnVy84ua9pthSjznHXLjDpZxbLe4WPRAQCT+LrSVcGGdLzNX0XKhesC2eV5uZ67lQPDlmSzUhS2+61ELtoTOyWGjdPudc73fvnj7c/Hg7ElpyzYXjdwIZ32m7X3INZ0crtvtc833ua6fyoUYUGf4H13rJpX+GK5aa5VbeWO9ye/03bLi97i8SpaSCl49nc4gdxI2p1dZyVmQnfL56jBH/j70GqSq6dyMROLHQGvCpcSnkYsyeohNErnVjbamVjs47wOpBa7BAsa61SXulDfSIwAbAkQqDmSK38WwImKK1kFJ3d11CpFqR1mPHlj1pWVAHeDfyU2hk0vGogz0GqCBbKmFMx9xIWEAbQ8I4PUvmAplQCeuij7GNXGMk3yhBsFIfEnvVE6HFYMHDtFuLzKwpya9gisaZduAkcyAvmw1ROjmd7OnBYytpOBqJjTyK4KzT0Pi0tQJYslc0nGqcNZV7dEaRvfcSewg9h4p41iwO+4CWMkNG1326VBmHFUoB7VoakzE+JAtYN/NknS9lLJ+9S5qR56Q2kLA4qgTuplGNmUJAkbXiGbrU0HCIsgtYaGOUVXbayNeykLfpGv8lulBfgiFEmx61xm3LTlrOPoa51Ng7IyJiT4tWxrE0ZmnJRnhawZsyLgjXIC9MRu0tRAgImkJ7bUyHKk1eUgIewtRjXGDJqO1mNJrHfNUty1HRW2SSoV3FAVIA//hrUXKANxhbRbEkgqAFKiCC43qOEXsPTaKtKCdkqx3/koMYsuP+mh4HrHoQtqSIw/h4DM7EpRL5zRirAcHiUDj2SUlr9fFAsElHBX/zWgJCQlw+83Qksw8Pt9+Ty0hmOBQBh9XQAr7fxjRQ2IkGTT/w8cAiBKwRc1jwcMh+WKtMgFShNaDGJWRAechNSOMUldXTynNIUPAaEKKkCv1cLFzxaowBOmX8vU8Pj1voWa5JTPFcGN4QXm+PpZPjA8QQYYp9Qab9rZiIAuLX8AA8f/jX4kHgI+BXCtBEeOTFvNPapIyC92YAEOmHypJcCSiFOpQi712M73IL8KiBX8Hz62pDNsAH8MK/rMR+tNSRIRJwAkIIY8Rn/fD2kB2V9I7BMX+KSJ/XJtNAbwFglmQZbu/90CZcMhAeZKudKAvlSBLwNwP1aA88BYpPdJRkbADGeBzWN2IOvySVELxyU0Lx0JHQHsFpEQRsCB9iO1qT3IOHGUjMDqeczX4BiIbvCjkiLD6t6G239p+gAML1KQuAdaLLdidDV6Y6uPR+9+3LT8KrlYgzAsjg7ok+VJakimvgAjn5qUFmtTGJKXGxSadg2UuLkWok4EvGJ0Hdsr+DmpVlGsUMZkxtuTI8vDAVaIsnSIDbK0DhVSpAFlu4iRtgrLYWRaArADOcFDAfErEdwBS8dWpNqHupW3o7nIukTmxlmASq4L/51ESrznqJTSkgCzslVSMncekb8659ljeyYttxK0oX9k50n3h2kDqoapToWt8S9/yO3tTXyteLu+19rkPViKkIblLnzJhJUhYENUJCtdfWKkEFe9WTsWin6azpqJFIfEwNyLNes+PZXFV3h+tIOXjb5mzIRnCLaWKuhnPtjsCVPAOwdkhQhBSMfWLVruTgwEM/WXt1FYgv5AN4IsyBiHrOSscCBjIomksgZCPFn9grqDrE9UXDQCSPfs5RXyeuQl3gwcEIdn5JzADhpG34s4uF5NQ24eh1GWISkEmYA6iFiNezYAbkKNGJhIk+l9XNXK3r7AgLT4YnRUuHicJS0NRLLG0KDrF1IbkaMzj2MoeKzAH/kTo9KsnWJe8gDZUuxs1iPi0CayABMdLIT4qQCbfENfCiAsujIsj9KMUQ1kXwXUBFZspHZm9Zj/dB/SrVy1qOJg9ApKAlmSQNpr6SjibnNgVoqZQCr8gllgsTiFAOCFSeDcYWuKOChZQXi5dPxouF5Oo6ogVTaxDwnVE8KfQFI6Zomwgb/oI4VG2Ae1MdsTQCwQuZp5ZROZPcdudWHrZRfwe4cPE1cv9L/FDuwRwGNYOttGllMRItS1K3sFDQDAwQMvgpfIzyIeQj2yTFOhomGLsBVPtAASGLqiXKnChASW/4ILUTGlftQlUVl8FDjifbKYIzLKXmco4anNPK6ZVl8MhVBs+D5YgTo8HXuIEDQDJF4wHEYL5PvEExn4PIt7x0JumDeUgjxHeFIZDt+6xN5mALCc6pMjTZ7oy8EomCUA09MTRvvuCLxyFC+sEWCGiqjE+HIAJ4g39Bi5sadSN9MJZaFWLbpubBNBvlmwrPAONJWS1DpaIgbyKuZSbKg7qZzgNKsqnwIX8RAOvINdgfRg0jPNJBWE+5xAQ9qDaAPh4nKInagjAL9otj26U+sAnbUERTF0QYjHV8j8OEA+mohtH9SLLuoUIh6uAWHI6yAw54uEsqL6zufMAAt3yW+6xSFLmKwEmYgIPlXHYbPAo3AvKovCk/0KR5m3fmKkpFz2W3ng58h7KzqVVHqVYUpfGlAJHvQ2dN7QKmXoQITggHUz9HGZhSty5smYHeSL4pFdvbXlXZcSJV7GCHKPpsePEmVAOm41WxbuW3Y+qkrhYhisbrSBqbGiGdAkBwd5gzNehxGUVhzu5JCKa2VAyL2pfYgYgUtcTwu3RKZzO2JublhfCDbO1Qqypu8SdTWO0501Z6iAAH68g21mECvjX8bZZ7yvpViqq9BlHCnhj7DFuiXm8qEwgrAkK9hpfbuU/nN/i4l7I/qQEnpR4qVHUxgDjblWuM2UZFhOpzp+apb8i4ua8gnyHrqPMHs839gK7gaaYDDtILqNTWOF8kxVYdMxlVDwE68/F8DR2CgpCSkkzkEvKY3x9GoRqjpn8ZLj62TyEoRVV1dxrV2GtNOMosPAjqSC6pm6yRKRJFM5wnixbhh6uLl9FdjElsUvX7WxR61T1gGohOkQaej27MxiR+DQjAndQ5SgL4Yb+VMKT2UncGK4se5hfeR7Jr6nygbTJcPGK7QSpXOoBJkuMXlFTYQX0aRsObYEQn22BNrDQdKwmZ1DDaZNcANxumdkGs3pZIJcaCcxBuPSlGziTf+UNZIgqmLMTWS1/XYNBxFowH5FchZb7uT5EQqervUK+Rc15pP55mBpKra6Wju5MCbZwiKOSS/HFuZyVRBFeeIoVrc35oNnVt1ARpKDaHF2BO1z3DLILOrOVZdhIG52ojyFxsVa1k6Bq+sOS7AA0upIqpiCt9Om8+1SsIoI4/ZYFKZq9tdwnhCzrVRpoCxpqSo+0ua1DNLCOMc3X19vGSZZ9znfEAMTUpymSGqYW/kpXU6FWV6+pah9OGLveaTkopOV1d3U9oWfAhMiyK46fRgPeL9LTSDAtUjj5cFN4D8S5zmaCjmaTjJ/Kvxb3MaFrHVI27QS2vRfcMCmeqcUXY+NX6652okyj19OEC3SaFrdWyeyJMxo7gPrANn17GjQ5RgtqvregNjzr1hYOOxATeos4b/IItGRxEFZC4aNruVL1ZbGx8ojpE5moLiGNavazB+baxPoXr/gK56zjI1VGbjtG8nA3Xg3SpFl2gtqzqsM/oGgb5Axj4w+XD1g48MBFrMAnR+TRxMci1+8h+uCC1e1nmC7XEWhf2zEdIw5SsCe3Tcc18XibsU/PKhJhd7a380s67RLEvQQWUw9KKjmps7qefVeub/IZQoLbK0qxnHRFP7qlOy8BURC7Fy2LDPk7N1F1VEm1lrZbSmSWVSoNk63DQXgvIohAjYcjXU7b/zI8u7RVvPiRChSqVjkiU6YQjrT1ImVAAN/WoKEp62V2aQGAdAuQKhSW5qouyIdQ4jNe5NS6I8v10hLIeqIIMqplDn71o0dYl4fEFkRZ4LmhFJNUSQqZCNlBT2fIWnKzJ9XWwG48bOzGZWEIffcVx/q23xAziBV83gf1cE4/+mvI8/EFV3QVmPW0a6rB7ZD314ploTllOSlFYqX3X4u7TJg6jmxLRWxit1FaPtArKoHfHeb3jwPmNGDqfvS+Iv9Ns14Vv6p9rfyft+HGiFqmJIQSU++rlbegPBqDumli9zoOGSptec8O6GAWqtaAgkOgqZ3P1WhQQ08aj2KGONuEIqdZetzuLFB6qQniid8HkZTnlyGsPGnA4lY5Qu1456WnbokG9IupwU0NoPhHTwRRUqynZU0HObV8QUyZuo/lb6tFhsdxb7bCoU9qWe8vLxBwj2laVzgS+bIZGMfee1Qaldl87gBZ5jjr3Y4rq2Xaf3LatohHwxzbqBKtv4d8bHnh1eUqfVRx07jc4zfzySrj8IGV9NMFX4mBKrq6FXc4Jz154/3737u7++fbxw+3Pz9N7eg0X0mHCeHfHbHrNhrCIpdXRA/LRRC4WR9sU/ZqJB46XJsJouwU1rPT+il6uKHqNAdbJ2InUJp0wVWBV7w8NuldU7uMyajZqh2N6UfeoM6ym1zLGizF6T6AnNQZiHO/KZMijZMOV2/yKQFKv0TSXq0Hb5teE9F4HLp50QKJ3OX64edZ7ie+++PLrd7t339z+5e7mx9/88Qt+f82dx5f//Omr6e8fvv/+49Pdwz0/f3/z19vH3z7x68uH++fpKhP+/Pjw/PDh4cfv+Hz86f5Jk99/93T7eHersfff/fnmh/H3y4fH8feLv9/x96ub5+l7vq9f0wj9msf8+HH6fhnDr3kMvzRGG3p8+PizVt3vaXy/ysjox5sPzx8fbxn+7cuWv7m9v3v68PFpsfH9pcWwLc1oyEI3f/7H/cPf7p7vnhZ6ev/+X/8GYIe3xg=='
_RAD_CAL_W = 0.40
_RAD_TOKEN_DIM, _RAD_HEAD_DIM = (2048, 512)
_RAD_E11_SLOTS = [('SAG_NOFS', 'Sagittal', None, False), ('COR_NOFS', 'Coronal', None, False), ('AX_NOFS', 'Axial', None, False), ('SAG_FS', 'Sagittal', None, True)]
_RAD_E11_CROP_MM = 130.0
_RAD_E13_SLOTS = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True), ('SAG_NOFS', 'Sagittal', None, False)]
_RAD_E13_CROP_MM = 130.0
_RAD_E13_CACHE_SLICES = 8
_RAD_E13_IMG = 224
SLOTS = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True)]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8

def _rad_sha256(path, chunk=8 << 20):
    digest = _rad_hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk), b''):
            digest.update(block)
    return digest.hexdigest()

def _rad_find_file(name, expected_sha=None, explicit_env=None):
    files = {_RAD_ENCODER_SHA256: ASSET / 'resnet-50-radimagenet-marwan/ResNet50.pt', _RAD_REFERENCE_HEADS_SHA256: ASSET / 'rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt', _RAD_E13_HEADS_SHA256: ASSET / 'kernel-sources/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt'}
    path = files.get(expected_sha)
    if path is None or not path.is_file():
        raise FileNotFoundError(name)
    if _rad_sha256(path) != expected_sha:
        raise RuntimeError(f'hash mismatch for {path}')
    return path

class _RadEncoder(_rad_nn.Module):

    def __init__(self):
        super().__init__()
        self.backbone = _rad_nn.Sequential(*list(_rad_resnet50(weights=None).children())[:-2])

    def forward(self, image):
        return self.backbone(image).mean(dim=(2, 3))

class _RadHead(_rad_nn.Module):

    def __init__(self):
        super().__init__()
        self.project = _rad_nn.Sequential(_rad_nn.LayerNorm(_RAD_TOKEN_DIM), _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM), _rad_nn.GELU())
        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * 0.01)
        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * 0.01)
        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.attn = _rad_nn.MultiheadAttention(_RAD_HEAD_DIM, 8, dropout=0.1, batch_first=True)
        self.fuse = _rad_nn.Sequential(_rad_nn.LayerNorm(_RAD_HEAD_DIM * 4), _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM), _rad_nn.GELU(), _rad_nn.Dropout(0.15))
        self.weight = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * 0.02)
        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), N_SLOT, CACHE_SLICES, _RAD_HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token, key_padding_mask=key_padding, need_weights=False)[0]
        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator
        mean = mean.expand(-1, len(_RAD_LABELS), -1)
        fused = self.fuse(_rad_torch.cat([attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias

def _rad_load_public_heads(device, expected_sha):
    heads_path = _rad_find_file('v52_radimagenet_heads.pt', expected_sha)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=True)
    expected = {'version': 'v52-radimagenet-resnet50-official-1', 'targets': _RAD_LABELS, 'encoder_sha256': _RAD_ENCODER_SHA256, 'encoder_source_commit': '0ce16f7375db4236e646829d1eca61cdb4282133', 'img': 224, 'slices_per_plane': 8, 'feature': 'global_average_pool'}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'public-v15 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('public-v15 bundle requires exactly five heads')
    if sorted((int(record.get('fold', -1)) for record in folds)) != list(range(5)):
        raise RuntimeError('public-v15 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return (heads, str(heads_path))

def _rad_load_e13_heads(device):
    heads_path = _rad_find_file('v52_e11_heads.pt', _RAD_E13_HEADS_SHA256)
    payload = _rad_torch.load(heads_path, map_location='cpu', weights_only=False)
    expected = {'version': 'e11-radimagenet-resnet50-diverse-1', 'targets': _RAD_LABELS, 'encoder_sha256': _RAD_ENCODER_SHA256, 'slots': [list(slot) for slot in _RAD_E13_SLOTS], 'crop_mm': _RAD_E13_CROP_MM, 'img': _RAD_E13_IMG, 'slices_per_plane': _RAD_E13_CACHE_SLICES, 'feature': 'global_average_pool'}
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f'E13 head contract drift for {key}')
    folds = payload.get('folds')
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError('E13 bundle requires exactly five heads')
    if sorted((int(record.get('fold', -1)) for record in folds)) != list(range(5)):
        raise RuntimeError('E13 fold identity drift')
    heads = []
    for record in folds:
        head = _RadHead().to(device).eval()
        head.load_state_dict(record['state_dict'], strict=True)
        heads.append(head)
    return (heads, str(heads_path))

@_rad_torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask, device):
    n, slots, slices, height, width = pixels.shape
    features = _rad_np.zeros((n, slots * slices, _RAD_TOKEN_DIM), _rad_np.float16)
    token_mask = _rad_np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = _rad_np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, height, width)
    batch = 192 if device.type == 'cuda' and _rad_torch.cuda.device_count() > 1 else 96 if device.type == 'cuda' else 8
    for start in range(0, len(valid), batch):
        indices = valid[start:start + batch]
        image = _rad_torch.from_numpy(flat[indices]).to(device).float().div_(127.5).sub_(1.0)
        image = image.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        amp = _rad_torch.autocast('cuda') if device.type == 'cuda' else _rad_contextlib.nullcontext()
        with amp:
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not _rad_np.isfinite(values).all():
            raise RuntimeError('V36 non-finite RadImageNet feature')
        features.reshape(-1, _RAD_TOKEN_DIM)[indices] = values.astype(_rad_np.float16)
    return (features, token_mask.astype(_rad_np.float32))

@_rad_torch.inference_mode()
def _rad_predict_head(head, features, masks, device, batch=64):
    predictions = []
    for start in range(0, len(features), batch):
        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)
        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)
        amp = _rad_torch.autocast('cuda') if device.type == 'cuda' else _rad_contextlib.nullcontext()
        with amp:
            predictions.append(_rad_torch.sigmoid(head(image, mask)).float().cpu())
    return _rad_torch.cat(predictions).numpy()

def _rad_rank_columns(values):
    return _rad_pd.DataFrame(_rad_np.asarray(values, dtype=_rad_np.float64)).rank(method='average', pct=True).to_numpy(_rad_np.float64)

def _rad_validate(frame, expected_ids):
    if frame.columns.tolist() != ['StudyInstanceUID', *_RAD_LABELS]:
        raise RuntimeError('V36 submission schema drift')
    ids = frame['StudyInstanceUID'].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError('V36 submission study identity/order drift')
    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)
    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError('V36 invalid submission values')


def _v18_cal_protocol(uids):
    frame = _rad_pd.read_csv(
        ROOT / 'test_series.csv',
        dtype={
            'StudyInstanceUID': str,
            'SeriesInstanceUID': str,
        },
    )

    frame['StudyInstanceUID'] = (
        frame['StudyInstanceUID'].astype(str)
    )

    index = _rad_pd.Index(
        [str(uid) for uid in uids],
        name='StudyInstanceUID',
    )

    table = _rad_pd.DataFrame(index=index)

    table['n_series'] = (
        frame.groupby('StudyInstanceUID')
        .size()
        .reindex(index)
        .fillna(0)
    )

    for plane in (
        'Sagittal',
        'Coronal',
        'Axial',
    ):
        part = frame[
            frame['Anatomical_Plane']
            .astype(str)
            .eq(plane)
        ]

        table[f'n_{plane[:3]}'] = (
            part.groupby('StudyInstanceUID')
            .size()
            .reindex(index)
            .fillna(0)
        )

    for flag in (
        'Fat_Suppression',
        'Fluid_Sensitive',
    ):
        marked = frame[
            _rad_pd.to_numeric(
                frame[flag],
                errors='coerce',
            )
            .fillna(0)
            > 0
        ]

        prefix = flag[:3]

        table[prefix] = (
            marked.groupby('StudyInstanceUID')
            .size()
            .reindex(index)
            .fillna(0)
        )

        for plane in (
            'Sagittal',
            'Coronal',
            'Axial',
        ):
            part = marked[
                marked['Anatomical_Plane']
                .astype(str)
                .eq(plane)
            ]

            table[f'{prefix}_{plane[:3]}'] = (
                part.groupby('StudyInstanceUID')
                .size()
                .reindex(index)
                .fillna(0)
            )

    return table


def _v18_calibrate_transformer(
    branch,
    baseline_rank,
    public_rank,
    pass2_rank,
    expected_ids,
):
    payload = _rad_json.loads(
        _rad_zlib.decompress(
            _rad_b64.b64decode(
                _RAD_CAL_PAYLOAD
            )
        ).decode()
    )

    gate = set(payload['gate'])

    protocol = _v18_cal_protocol(
        expected_ids
    )

    if (
        protocol.columns.tolist()
        != list(
            payload[
                'protocol_columns'
            ]
        )
    ):
        raise RuntimeError(
            'V18 calibration protocol '
            'layout mismatch'
        )

    mean_rank = (
        baseline_rank
        + public_rank
        + pass2_rank
    ) / 3.0

    blocks = [
        baseline_rank,
        public_rank,
        pass2_rank,
        public_rank - baseline_rank,
        pass2_rank - baseline_rank,
        mean_rank,
    ]

    for group in payload['groups']:
        columns = [
            _RAD_LABELS.index(target)
            for target in group
        ]

        blocks.append(
            mean_rank[
                :,
                columns,
            ].mean(
                axis=1,
                keepdims=True,
            )
        )

    blocks.append(
        protocol.to_numpy(
            _rad_np.float64
        )
    )

    x = _rad_np.concatenate(
        blocks,
        axis=1,
    )

    centre = _rad_np.asarray(
        payload['mean'],
        _rad_np.float64,
    )
    spread = _rad_np.asarray(
        payload['scale'],
        _rad_np.float64,
    )
    coef = _rad_np.asarray(
        payload['coef'],
        _rad_np.float64,
    )
    bias = _rad_np.asarray(
        payload['intercept'],
        _rad_np.float64,
    )

    if (
        x.shape[1] != 88
        or coef.shape != (
            len(_RAD_LABELS),
            88,
        )
    ):
        raise RuntimeError(
            f'V18 calibration feature drift: '
            f'x={x.shape}, coef={coef.shape}'
        )

    spread = _rad_np.where(
        _rad_np.abs(spread) > 1e-8,
        spread,
        1.0,
    )

    adjusted = _rad_rank_columns(
        (
            (
                x - centre
            )
            / spread
        )
        @ coef.T
        + bias
    )

    output = branch.copy()

    values = output[
        _RAD_LABELS
    ].to_numpy(
        _rad_np.float64
    ).copy()

    for index, target in enumerate(
        _RAD_LABELS
    ):
        if target in gate:
            values[
                :,
                index,
            ] = (
                (
                    1.0
                    - _RAD_CAL_W
                )
                * values[
                    :,
                    index,
                ]
                + _RAD_CAL_W
                * adjusted[
                    :,
                    index,
                ]
            )

    output[
        _RAD_LABELS
    ] = _rad_rank_columns(
        values
    )

    _rad_validate(
        output,
        expected_ids,
    )

    return output, gate



def _rad_main():
    work = _RadPath('/kaggle/working')
    primary = work / 'submission.csv'
    test = _rad_pd.read_csv(ROOT / 'test.csv', dtype={'StudyInstanceUID': str})
    expected_ids = test.StudyInstanceUID.astype(str).tolist()
    baseline = _rad_pd.read_csv(primary, dtype={'StudyInstanceUID': str})
    _rad_validate(baseline, expected_ids)
    device = _rad_torch.device('cuda:0')
    test_series = _rad_pd.read_csv(ROOT / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
    plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))

    def cache(slots, crop, tag, threshold):
        globals().update(SLOTS=list(slots), N_SLOT=len(slots), CACHE_SLICES=8, IMG=224, CACHE_IMG=224, CROP_MM=float(crop), RULES=dict(RULES_LEGACY))
        headers = annotate(walk('test_series'))
        studies, pixels, masks = build_cache(pick_slots(headers, plane), plane, lat_of(headers, tag + ' '), tag)
        positions = {str(uid): index for index, uid in enumerate(studies)}
        missing = [uid for uid in expected_ids if uid not in positions]
        if missing:
            raise RuntimeError(f'{len(missing)} studies absent from {tag}')
        order = _rad_np.asarray([positions[uid] for uid in expected_ids], dtype=_rad_np.int64)
        pixels, masks = (pixels[order], masks[order])
        tokens = int(_rad_np.repeat(masks[:, :, None], CACHE_SLICES, axis=2).sum())
        if tokens < int(threshold * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f'insufficient slices for {tag}: {tokens}')
        return (pixels, masks)
    public_slots = [('SAG_FS', 'Sagittal', None, True), ('COR_FS', 'Coronal', None, True), ('AX_FS', 'Axial', None, True)]
    pixels, masks = cache(public_slots, 10000.0, 'test-e10', 0.85)
    encoder_path = _rad_find_file('ResNet50.pt', _RAD_ENCODER_SHA256)
    encoder = _RadEncoder()
    encoder.load_state_dict(_rad_torch.load(encoder_path, map_location='cpu', weights_only=True), strict=True)
    encoder.eval().to(device)
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    if _rad_torch.cuda.device_count() > 1:
        encoder = _rad_nn.DataParallel(encoder, device_ids=list(range(_rad_torch.cuda.device_count())))
    reference_heads, _ = _rad_load_public_heads(device, _RAD_REFERENCE_HEADS_SHA256)
    features, token_mask = _rad_encode(encoder, pixels, masks, device)
    reference_predictions = [_rad_predict_head(head, features, token_mask, device) for head in reference_heads]
    reference_probability = _rad_np.mean(_rad_np.stack(reference_predictions), axis=0)
    reference_rank = _rad_rank_columns(reference_probability)
    del reference_predictions, reference_heads
    del reference_probability, features, token_mask, pixels, masks
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()
    globals().update(SLOTS=list(_RAD_E13_SLOTS), N_SLOT=len(_RAD_E13_SLOTS), CACHE_SLICES=_RAD_E13_CACHE_SLICES, IMG=_RAD_E13_IMG, CACHE_IMG=_RAD_E13_IMG, CROP_MM=_RAD_E13_CROP_MM, RULES=dict(RULES_LEGACY))
    e13_heads, _ = _rad_load_e13_heads(device)
    pixels, masks = cache(_RAD_E13_SLOTS, _RAD_E13_CROP_MM, 'test-e13', 0.85)
    features, token_mask = _rad_encode(encoder, pixels, masks, device)
    e13_predictions = [_rad_predict_head(head, features, token_mask, device) for head in e13_heads]
    e13_probability = _rad_np.mean(_rad_np.stack(e13_predictions), axis=0)
    e13_rank = _rad_rank_columns(e13_probability)
    reference_rank = _rad_rank_columns((1.0 - _RAD_E13_MEMBER_WEIGHT) * reference_rank + _RAD_E13_MEMBER_WEIGHT * e13_rank)
    del e13_predictions, e13_probability, e13_rank
    del features, token_mask, pixels, masks
    _rad_gc.collect()
    _rad_torch.cuda.empty_cache()
    baseline_rank = _rad_rank_columns(baseline[_RAD_LABELS].to_numpy())
    e10 = baseline.copy()
    for index, target in enumerate(_RAD_LABELS):
        if target not in _RAD_EXCLUDE:
            e10[target] = (1.0 - _RAD_ALPHA) * baseline_rank[:, index] + _RAD_ALPHA * reference_rank[:, index]
    _rad_validate(e10, expected_ids)
    pixels, masks = cache(_RAD_E11_SLOTS, _RAD_E11_CROP_MM, 'test-v48-pass2', 0.55)
    features, token_mask = _rad_encode(encoder, pixels, masks, device)
    pass2_predictions = [_rad_predict_head(head, features, token_mask, device) for head in e13_heads]
    pass2_probability = _rad_np.mean(_rad_np.stack(pass2_predictions), axis=0)
    pass2_rank = _rad_rank_columns(pass2_probability)
    final = e10.copy()
    final[_RAD_LABELS] = (
        (1.0 - _RAD_V48_SECOND_ALPHA)
        * _rad_rank_columns(
            e10[_RAD_LABELS].to_numpy()
        )
        + _RAD_V48_SECOND_ALPHA
        * pass2_rank
    )

    final[_RAD_LABELS] = _rad_rank_columns(
        final[_RAD_LABELS].to_numpy()
    )

    _rad_validate(
        final,
        expected_ids,
    )

    globals()['V18_TRANSFORMER_RAW'] = (
        final.copy()
    )
    globals()['V18_CALIBRATOR_APPLIED'] = False
    globals()['V18_CAL_GATE'] = tuple()

    try:
        calibrated, gate = (
            _v18_calibrate_transformer(
                final,
                baseline_rank,
                reference_rank,
                pass2_rank,
                expected_ids,
            )
        )

        final = calibrated

        globals()[
            'V18_TRANSFORMER_CAL'
        ] = final.copy()

        globals()[
            'V18_CALIBRATOR_APPLIED'
        ] = True

        globals()[
            'V18_CAL_GATE'
        ] = tuple(
            sorted(gate)
        )

        print(
            '[V18] 88-feature transformer '
            'calibration applied to: '
            + ', '.join(
                sorted(gate)
            ),
            flush=True,
        )

    except Exception as exc:
        print(
            '[V18] calibration skipped '
            'safely; raw transformer kept: '
            f'{type(exc).__name__}: {exc}',
            flush=True,
        )

    _rad_validate(
        final,
        expected_ids,
    )

    final.to_csv(
        primary,
        index=False,
    )
_rad_main()


## セル6：Raptor 4視点確率アンサンブル V40 と最終ブレンド

**何をしているか**：Raptorモデルの4つのアーム（`v5=.55, v10=.10, reverse-v5=.15, v8=.20`、外側の重み `outer=0.60`）を重み付き平均し、最後にCoAtNetの出力とランクでブレンドして `submission.csv` を確定させます。

そして冒頭のdocstringに、このコンペで最も重要な情報が書かれています。

### 弱ラベル（weak label）の作り方

> **4,407件のstudyのうち、構造化ラベルが付いているのは58件だけ。** 残りは自由記述の放射線科レポートしかない。

そこで、**レポートを言語モデルに読ませて12個の確率に変換**しました。ポイントは：

- 「断裂が**疑われる**（suspected）」というレポートは、**1ではなく0.8** になる。
- 断定的な記述は1に近く、否定は0に近く、曖昧な記述は中間に置かれる。

これで学習可能なstudyが **58 → 4,349件** に増えました。

**なぜそうするのか**：

- **「曖昧さを曖昧さのまま目的変数にする」**——これが弱ラベルの核心です。「疑われる」を無理に1にすると、モデルは「疑わしいだけの症例も断裂だ」と学習してしまい、確信度の分布が歪みます。0.8を目標にすれば、モデルは「確信は持てないが可能性は高い」という状態を素直に学べます。
- 二値ラベルの代わりにソフトラベルで学習するのは、**知識蒸留（knowledge distillation）**と同じ発想です。「正解は1つ」という情報より、「1に近いが完全ではない」という情報の方が、はるかに多くを教えてくれます。
- **58件で学習するのは事実上不可能です。** 12所見 × 深層モデルのパラメータ数を考えれば、58件は絶対的に足りません。弱ラベルは「精度の高いラベルが少しある」より「多少ノイズがあるラベルがたくさんある」方が勝つ、という現代的な選択です。

### 4アームの重み

`v5=.55, v10=.10, reverse-v5=.15, v8=.20`。合計1.0。`reverse-v5` はスライス順を逆にした版で、これも一種のTTAです。重みは手調整（LB由来）であることに注意——**過学習のリスクがあります**（READMEの改善提案参照）。

**用語補足**：*弱教師あり学習（weak supervision）* = 完全なラベルの代わりに、ノイズを含む・不完全な・間接的な信号で学習すること。医療分野ではラベル付けに専門医の時間が必要なので、極めて重要な技術です。

In [ ]:
# RAPTOR_FOUR_VIEW_PROBABILITY_ENSEMBLE_V40
# Global weights: v5=.55, v10=.10, reverse-v5=.15, v8=.20; outer=0.60
#!/usr/bin/env python3
"""Knee MRI: twelve findings from a single model

This notebook takes a knee MRI study and scores twelve findings at once: ACL tear, MCL tear,
medial and lateral meniscus tears, osteoarthritis in the medial, lateral and patellofemoral
compartments, joint effusion, synovitis, a Baker's cyst, bone contusion and fracture. It scores
0.924 on the public leaderboard using one model, with no ensembling and no test-time augmentation.

This is the inference half of the work. The model was trained separately and its weights are
attached as a dataset, so this notebook only loads them and predicts:
https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense

Where the training labels came from

Worth saying up front, because it shapes everything else. The competition gives you 4,407 studies
but structured labels for only 58 of them. Every other study arrives with a free-text radiology
report and nothing more, so there is very little to train against out of the box.

The labels behind these weights were made by reading those reports with a language model and
turning each into twelve probabilities rather than twelve yes or no answers. A report that says a
tear is suspected becomes a number near 0.8, not a 1, which is a fairer target than forcing every
hedged sentence into a hard label. That yields 4,349 studies to train on. The 58 studies that came
with real labels were never trained on and are used to check the result honestly; the model reaches
0.9167 macro-AUC on them.

Building a fixed input from studies that are all shaped differently

The hard part of this competition is not the network, it is that no two studies look alike. A
study holds several DICOM series shot in different planes, the number of series varies, and the
number of slices in a series varies more. Anything that expects a fixed-size input has to be given
one.

The approach here is to fill five fixed slots per study, always in the same order, for a stack of
64 images:

  18 slices from a sagittal series, preferring a fluid-sensitive one
  14 slices from a second sagittal series, preferring one that is not fluid-sensitive
  12 slices from a coronal series, preferring a fluid-sensitive one
   8 slices from a second coronal series
  12 slices from an axial series

Preferring a fluid-sensitive series for some slots and not for others is deliberate. Fluid-
sensitive sequences show swelling, effusion and acute injury clearly, while the other sequences
show anatomy and cartilage better, and the twelve findings are split across both. If a study has
no series for a slot, the slot is left as zeros and the model is told to skip it rather than being
fed something misleading.

Within a series, slices are taken evenly across 6 to 94 percent of the stack rather than from the
middle. The outer slices are where the collateral ligaments and the lateral meniscus sit, and
cutting them was measurably costing accuracy on exactly those findings.

Every slice is cropped to a 140 mm box around the centre of the image using the pixel spacing from
the DICOM header, then resized to 336 pixels. Cropping by millimetres rather than by pixel count
matters: it means a knee occupies the same fraction of the frame whether the scan was acquired at
0.3 or 0.5 mm per pixel, so the model is not asked to learn scale differences that carry no medical
information.

How the model reads the stack

Three neighbouring slices are stacked into the three channels of one image. The network then sees
a little of what lies above and below the slice in the middle, which is most of the benefit of a 3D
model at the cost of a 2D one. Each of these three-slice windows is passed through a CoAtNet
backbone at 384 pixels.

The windows are combined with an attention layer that has separate weights for each of the twelve
findings. This is the part that matters most. A cruciate tear may be visible on two sagittal slices
while osteoarthritis is spread across many coronal ones, and a single pooled score forces those two
to share one notion of which slices are important. Giving each finding its own attention weights
lets each one draw on the slices that actually show it.

Running it

Scoring uses 42 windows per study. Inference runs in half precision and automatically retries a
study in full precision if it fails, so no study is ever dropped from the submission. The notebook
needs no internet: the backbone is loaded from the attached weights rather than downloaded.
"""
import os, sys, glob, time, json, gc
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
# T4 (Turing) cuDNN v9 has fp16/fp32 conv engines but NOT bf16 for these shapes
# ("GET was unable to find an engine..."); benchmark lets it pick a valid algo for
# the fixed (1,24,3,res,res) input.
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- fixed config (must match training exactly) -----------------------------
# Defaults are overwritten from each arm's immutable pixel contract before inference.
IMG = 336
CROP_MM = 140.0
SPAN_LO, SPAN_HI = 0.02, 0.98
SLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
         ("Coronal", 0, 8), ("Axial", -1, 12)]
MAXS = sum(s[2] for s in SLOTS)
K_EVAL = 62
NORM = "imagenet"
LAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
       "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Three arms: (weights filename, fallback arch, fallback res). ck carries arch+res too.
# Selected 2026-08-19 by greedy forward selection AND exhaustive subset search over a 7-arm
# panel on the 45-study gold set (phase2/blend_panel.py); both agree on this exact set.
# Singles: coatnet384 0.9025 | swinbase384 0.8825 | effv2l480 0.8716.
# Blend {coatnet+swin+effv2l} = 0.9068 (2-arm {coatnet+swin} = 0.9059, coatnet alone 0.9025).
# Dropped as redundant: cnn336 (0.8833, the former champion), cnbase384 (0.8754),
# cnlarge384 (0.8752), maxvit384 (0.8438).
#
# SINGLE ARM: coatnet_rmlp_2_rw_384 retrained on the EXPANDED 4,349-study corpus.
#
# Why one arm and not the 3-arm blend: on the live leaderboard CoAtNet alone scored 0.914 while
# every blend scored 0.914-0.915, so ensembling is worth ~+0.001 there -- the ~+0.010 it showed
# on the old 45-study gold set was gold-set noise. One arm is also 1/3 the kernel runtime.
#
# Corpus expansion: the corpus previously held 3,200 of the 4,349 labelled studies and only 45
# of the 58 gold studies. Rebuilt to 4,407 studies (+37.8% training data, 58-study gate).
#
# Measured on the 58-study gate (the incumbent re-scored on the SAME gate for a fair compare):
#   incumbent CoAtNet (3,155-study corpus) 0.8923
#   this model       (4,349-study corpus) 0.9054   (+0.0131, better in 92.7% of 2000 bootstraps)
# Biggest gains land on the findings that were capping us: Lateral Meniscus +0.071,
# Fracture +0.057, Lateral OA +0.048, Medial Meniscus +0.035, ACL +0.028.
# Four globally weighted views.  Weights and the 0.70 outer blend were frozen
# after the same configuration improved both Gold58 anchor constructions.  There is
# no per-target routing: every finding receives the same estimator.
_SLOTS64 = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
            ("Coronal", 0, 8), ("Axial", -1, 12)]
_SLOTS44 = [("Sagittal", 1, 12), ("Sagittal", 0, 10), ("Coronal", 1, 8),
            ("Coronal", 0, 6), ("Axial", -1, 8)]
ARMS = [
    {"name": "maxspan-v5", "file": "raptor_ft_coatnet_v5_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": False, "w": 0.55},
    {"name": "native384dense-v10", "file": "raptor_ft_coatnet_v10_full.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 384, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": False, "w": 0.10},
    {"name": "maxspan-v5-reverse", "file": "raptor_ft_coatnet_v5_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,
     "reverse": True, "w": 0.15},
    {"name": "native384-v8", "file": "raptor_ft_coatnet_v8_full_swa.pt",
     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,
     "img": 384, "slots": _SLOTS44, "span": (0.06, 0.94), "k_eval": 42,
     "reverse": False, "w": 0.20},
]


# ============================================================================
# Model -- verbatim from finetune_raptor.py
# ============================================================================
def build_backbone(arch, pretrained=False):
    # maxvit/maxxvit/coatnet are conv-attention hybrids: NO CLS token, NO interpolatable
    # pos-embed -> avg pool. The "vit" substring in "coatnet"/"maxvit" must NOT route them
    # down the ViT path (mirrors finetune_raptor.py exactly).
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = self.att(h)
        a = torch.softmax(a, dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        logits = (pooled * self.clsW).sum(-1) + self.clsb
        return logits

    def forward(self, x):
        return self.head(self.encode(x))


def load_model(pt_path, arch_default, res_default, device, ngpu=1):
    ck = torch.load(pt_path, map_location="cpu", weights_only=False)
    arch = ck.get("arch", arch_default)
    ck_res = int(ck.get("res", res_default))
    bb = build_backbone(arch, pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(device)
    # NOTE: DataParallel removed on purpose. On the full hidden test it drove a system-RAM OOM
    # (per-forward module replication over many studies); a single T4 handles K_EVAL=24 windows
    # fine. Arms are also run SEQUENTIALLY (see main) so peak RAM == one model, not two.
    del ck
    gc.collect()
    return model, ck_res


# ============================================================================
# Eval windowing -- verbatim from finetune_raptor.py StudyWindows (train=False)
# ============================================================================
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


def eval_windows(vol, mask, k, res, norm=NORM):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear",
                              align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    if norm == "imagenet":
        x = (x - _MEAN) / _STD
    return x


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(device)
    use_cuda = device != "cpu" and str(device).startswith("cuda")
    if use_cuda:
        # fp16 conv on T4 is fully cuDNN-supported (bf16 is NOT -> "no engine").
        try:
            with torch.autocast("cuda", dtype=torch.float16):
                o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
        except RuntimeError:
            # fp32 always has a Turing conv engine; slower but never drops a study.
            torch.cuda.empty_cache()
            o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
    o = torch.sigmoid(model(x).float())
    return o[0].cpu().numpy()


def rankpct(x):                                   # per-column percentile rank in [0,1]
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


# ============================================================================
# Preprocessing -- verbatim from kprep2/dino_preprocess.py, retargeted to TEST
# ============================================================================
def _make_reader():
    import pydicom, cv2
    from pydicom.pixel_data_handlers.util import apply_modality_lut

    def order_and_meta(sdir):
        fs = glob.glob(sdir + "/*.dcm"); recs = []; ps_list = []
        for f in fs:
            try:
                h = pydicom.dcmread(f, stop_before_pixels=True)
                iop = getattr(h, 'ImageOrientationPatient', None)
                ipp = getattr(h, 'ImagePositionPatient', None)
                if iop is not None and ipp is not None and len(iop) == 6:
                    r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                    n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
                else:
                    pos = float(getattr(h, 'InstanceNumber', 0) or 0)
                ps = getattr(h, 'PixelSpacing', None); ps = float(ps[0]) if ps is not None else 0.5
                ps_list.append(ps); recs.append((pos, f, ps))
            except Exception:
                recs.append((0.0, f, 0.5))
        recs.sort(key=lambda x: x[0])
        med_ps = float(np.median(ps_list)) if ps_list else 0.5
        return [(f, ps) for _, f, ps in recs], med_ps

    def read_px(f):
        d = pydicom.dcmread(f)
        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
        if str(getattr(d, 'PhotometricInterpretation', '')) == 'MONOCHROME1':
            a = a.max() - a
        return a

    def mm_crop_resize(a, ps):
        h, w = a.shape; cpx = int(round(CROP_MM / max(ps, 1e-3)))
        cpx = min(cpx, min(h, w)); y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
        a = a[y0:y0 + cpx, x0:x0 + cpx]
        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)

    return order_and_meta, read_px, mm_crop_resize


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r['Anatomical_Plane'] == plane and r['SeriesInstanceUID'] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get('Fluid_Sensitive', 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(sid, ser_records, tsdir, reader):
    order_and_meta, read_px, mm_crop_resize = reader
    rows = ser_records.get(sid, [])
    vol = np.zeros((MAXS, IMG, IMG), np.uint8); idx = 0; used = set()
    for plane, fluid, k in SLOTS:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k; continue
        used.add(r['SeriesInstanceUID'])
        files, med_ps = order_and_meta(f"{tsdir}/{sid}/{r['SeriesInstanceUID']}")
        if not files:
            idx += k; continue
        # wide span: the collateral ligaments and lateral meniscus live in the
        # peripheral slices the old 0.15-0.85 crop threw away. Must match the corpus
        # the weights were trained on (knee_corpus_v2.py, SPAN_LO/SPAN_HI).
        n = len(files); lo, hi = int(n * SPAN_LO), int(n * SPAN_HI) - 1; hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        arrs = []; pss = []
        for p in picks:
            fp, ps = files[min(p, n - 1)]
            try:
                arrs.append(read_px(fp)); pss.append(ps)
            except Exception:
                arrs.append(None); pss.append(med_ps)
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= MAXS: break
            if a is None: idx += 1; continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = mm_crop_resize(aw, ps if ps > 0 else med_ps)
            vol[idx] = (aw * 255).astype(np.uint8); idx += 1
        if idx >= MAXS: break
    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask


# ============================================================================
# Test-root discovery + weights + main
# ============================================================================
def find_test_root():
    cands = ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
             "/kaggle/input/rsna-knee-abnormality-detection"]
    for b in cands:
        if os.path.exists(b + "/test.csv"):
            return b
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f and (os.path.isdir(d + "/test_series") or os.path.isdir(d + "/test_images")):
            return d
    for d, _, f in os.walk("/kaggle/input"):
        if "test.csv" in f:
            return d
    raise RuntimeError("no test root under /kaggle/input")


def find_weight_file(fname):
    # direct dataset mounts first; NEVER recursive-glob the competitions DICOM tree.
    direct = [f"/kaggle/input/raptor-knee-maxspan/{fname}",
              f"/kaggle/input/raptor-knee-native384dense/{fname}",
              f"/kaggle/input/raptor-knee-native384/{fname}",
              f"/kaggle/input/raptor-knee-arms/{fname}",
              f"/kaggle/input/raptor-knee-arms/1/{fname}",
              f"/kaggle/input/raptor-cnn336/{fname}"]
    for p in direct:
        if os.path.exists(p):
            return p
    for d in sorted(glob.glob("/kaggle/input/*/")):
        if "competition" in d.lower():
            continue
        hits = glob.glob(os.path.join(d, "**", fname), recursive=True)
        if hits:
            return hits[0]
    raise RuntimeError(f"{fname} not found under /kaggle/input")


def main():
    import pandas as pd
    t0 = time.time()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ngpu = torch.cuda.device_count()
    print(f"device {dev} | gpus {ngpu} | torch {torch.__version__}", flush=True)

    ROOT = find_test_root()
    tsdir = ROOT + "/test_series"
    if not os.path.isdir(tsdir):
        tsdir = ROOT + "/test_images"
    print("test root:", ROOT, "| series dir:", tsdir, flush=True)

    test = pd.read_csv(ROOT + "/test.csv"); test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
    test_ids = test["StudyInstanceUID"].tolist()
    tser = pd.read_csv(ROOT + "/test_series.csv")
    tser["StudyInstanceUID"] = tser["StudyInstanceUID"].astype(str)
    tser["SeriesInstanceUID"] = tser["SeriesInstanceUID"].astype(str)
    SER = {k: v.to_dict("records") for k, v in tser.groupby("StudyInstanceUID")}
    print(f"test studies {len(test_ids)} | test series {len(tser)}", flush=True)

    sub_cols = ["StudyInstanceUID"] + LAB
    ssub = os.path.join(ROOT, "sample_submission.csv")
    if os.path.exists(ssub):
        sub_cols = list(pd.read_csv(ssub, nrows=1).columns)

    reader = _make_reader()
    N = len(test_ids); A = len(ARMS)
    arm_probs = [np.full((N, len(LAB)), 0.5, np.float32) for _ in range(A)]

    # SEQUENTIAL ARMS (the OOM fix): only ONE model is resident at a time, so peak system RAM ==
    # one model == the single-arm champion's footprint (which graded fine at 0.879). Holding both
    # arms simultaneously OOM'd system RAM on the full hidden test. Each study is re-preprocessed
    # per arm (build_study is cheap vs inference) and every per-study buffer is freed. Same models,
    # same windowing, same rank-mean blend -> identical 0.8893 result, just serialized.
    for a, arm in enumerate(ARMS):
        # Restore the exact preprocessing contract used to train this checkpoint.
        globals()["IMG"] = int(arm["img"])
        globals()["SLOTS"] = list(arm["slots"])
        globals()["MAXS"] = sum(slot[2] for slot in SLOTS)
        globals()["SPAN_LO"], globals()["SPAN_HI"] = map(float, arm["span"])
        globals()["K_EVAL"] = int(arm["k_eval"])
        wp = find_weight_file(arm["file"])
        model, res = load_model(wp, arm["arch"], arm["res"], dev)
        print(f"[arm {a}] {arm['name']} | img {IMG} | slices {MAXS} | span {SPAN_LO:.2f}-{SPAN_HI:.2f} | windows {K_EVAL} | res {res} | {time.time()-t0:.0f}s", flush=True)
        for i, sid in enumerate(test_ids):
            try:
                vol, mask = build_study(sid, SER, tsdir, reader)
                xw = eval_windows(vol, mask, k=K_EVAL, res=res, norm=NORM)
                if bool(arm.get("reverse", False)):
                    xw = xw.flip(1).contiguous()
                arm_probs[a][i] = infer_probs(model, xw, dev)
                del vol, mask, xw
            except Exception as e:
                print(f"  [arm {a}] study {i} {sid[:16]} FALLBACK ({type(e).__name__}: {e})", flush=True)
            if (i + 1) % 100 == 0 or i + 1 == N:
                print(f"  [arm {a}] {i+1}/{N} | {time.time()-t0:.0f}s", flush=True)
        del model
        gc.collect()
        if str(dev).startswith("cuda"):
            torch.cuda.empty_cache()
        print(f"[arm {a}] done + freed | {time.time()-t0:.0f}s", flush=True)

    # WEIGHTED rank-mean blend across the test set, per finding (the offline recipe).
    # Weights come from ARMS[*]["w"] and are normalised here, so dropping/adding an arm can
    # never silently change the scale. Falls back to equal weights if none are declared.
    _w = np.array([float(a.get("w", 1.0)) for a in ARMS], dtype=np.float64)
    _w = _w / _w.sum()
    print(f"[blend] global probability mean w={dict(zip([a['name'] for a in ARMS], _w.round(4)))}", flush=True)
    probability_blend = np.tensordot(
        _w, np.stack([np.clip(p, 0, 1) for p in arm_probs]), axes=(0, 0)
    )
    ranks = rankpct(probability_blend)                                         # (N,12) in [0,1]
    if not np.isfinite(ranks).all():
        ranks[~np.isfinite(ranks)] = 0.5

    sub = pd.DataFrame(ranks.astype(np.float32), columns=LAB)
    sub.insert(0, "StudyInstanceUID", test_ids)
    sub = sub[sub_cols]
    assert list(sub.columns) == sub_cols, "column order drift"
    assert sub["StudyInstanceUID"].tolist() == test_ids, "row identity drift"
    assert np.isfinite(sub[LAB].values).all()
    out = "/kaggle/working/submission_coatnet.csv"
    sub.to_csv(out, index=False)
    print("wrote", out, "|", len(sub), "rows x", len(sub.columns), "cols", flush=True)
    print(sub.head().to_string(index=False), flush=True)
    print(f"DONE {time.time()-t0:.0f}s", flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as _coat_exc:
        import traceback as _coat_traceback
        print(f"CoAtNet branch failed; retaining transformer submission: {type(_coat_exc).__name__}: {_coat_exc}", flush=True)
        _coat_traceback.print_exc()



# Blend two independently validated rank predictors. The default remains the transformer
# submission if the CoAtNet branch did not complete, so a recoverable branch failure
# cannot erase a valid competition artifact.
from pathlib import Path as _BlendPath
import numpy as _blend_np
import pandas as _blend_pd

_blend_work = _BlendPath('/kaggle/working')
_blend_transformer_path = _blend_work / 'submission.csv'
_blend_coatnet_path = _blend_work / 'submission_coatnet.csv'
if _blend_coatnet_path.is_file():
    _blend_transformer = _blend_pd.read_csv(_blend_transformer_path, dtype={'StudyInstanceUID': str})
    _blend_coatnet = _blend_pd.read_csv(_blend_coatnet_path, dtype={'StudyInstanceUID': str})
    _blend_labels = [c for c in _blend_transformer.columns if c != 'StudyInstanceUID']
    if _blend_coatnet.columns.tolist() != _blend_transformer.columns.tolist():
        raise RuntimeError('CoAtNet/transformer submission schema mismatch')
    if _blend_coatnet['StudyInstanceUID'].tolist() != _blend_transformer['StudyInstanceUID'].tolist():
        raise RuntimeError('CoAtNet/transformer study order mismatch')
    _blend_tr = _blend_transformer[
        _blend_labels
    ].rank(
        method='average',
        pct=True,
    )

    _blend_cr = _blend_coatnet[
        _blend_labels
    ].rank(
        method='average',
        pct=True,
    )

    _blend_output = (
        _blend_transformer.copy()
    )

    # One global outer weight, fixed before the public submission.
    _coatnet_weight = {label: 0.60 for label in _blend_labels}

    for _label in _blend_labels:
        _cw = float(
            _coatnet_weight[
                _label
            ]
        )

        _blend_output[
            _label
        ] = (
            (
                1.0
                - _cw
            )
            * _blend_tr[
                _label
            ]
            + _cw
            * _blend_cr[
                _label
            ]
        )

    _blend_output[
        _blend_labels
    ] = _blend_output[
        _blend_labels
    ].rank(
        method='average',
        pct=True,
    )

    print(
        '[V18] CoAtNet target weights: '
        + ', '.join(
            f'{label}='
            f'{_coatnet_weight[label]:.2f}'
            for label in _blend_labels
            if (
                _coatnet_weight[label]
                != 0.50
            )
        ),
        flush=True,
    )

    _blend_values = _blend_output[
        _blend_labels
    ].to_numpy(
        _blend_np.float64
    )
    if not _blend_np.isfinite(_blend_values).all() or _blend_values.min() < 0 or _blend_values.max() > 1:
        raise RuntimeError('invalid blended prediction values')
    _blend_output.to_csv(_blend_transformer_path, index=False)
    print(f'final submission.csv = V18 calibrated transformer + CoAtNet rank blend; {_blend_output.shape}', flush=True)
else:
    print('CoAtNet output unavailable; submission.csv remains the validated transformer ensemble', flush=True)

# V18 output hygiene.
for _v18_temp in (
    _blend_work / 'submission_coatnet.csv',
    _blend_work / 'submission_transformer_0920.csv',
):
    try:
        if _v18_temp.is_file():
            _v18_temp.unlink()
    except OSError:
        pass


## セル7：実行時監査（append-only runtime audit）

**何をしているか**：**submission.csv には一切書き込まず**、実行が意図どおりだったかを検証してレシート（JSON）を残します。

検証内容：

- 親notebook本体、可視テストのUID一覧、親の提出物の **SHA256 が事前定義値と一致するか**
- RadImageNet較正のゲート対象が期待どおり7所見か（`ACL, Baker's, Contusion, Effusion, Lateral OA, Medial OA, PF OA`）
- Raptorの4アームの名前と重みが期待どおりか
- 12列のスキーマが崩れていないか

`submission_parent_exact.csv` として親の提出物をバイト単位でコピーしてから検証しています。

**なぜそうするのか**：

このnotebookの主張は「**親の0.936を一切変えず、内側半月板だけを差し替えた**」です。その主張が正しくなければ、報告された +0.001 の解釈が根本から変わってしまいます（「他の所見も動いていた」なら、内側半月板の寄与は測れていないことになる）。

**ハッシュ照合は、その主張の証明です。** 「変えていないつもり」ではなく「変えていないことをコードで確認した」。実験の信頼性を、言葉ではなく検証可能な形で担保しています。

これは前日までのBiohub notebookに出てきた `Configuration Guard` や `Pipeline Manifest` と全く同じ思想で、**別の人が別のコンペで独立に同じ結論に到達している**のが興味深い点です。上位のnotebookに共通するのは、モデルの新しさより**「自分の実験を信用できる状態に保つ仕組み」**だという傾向が、ここでも確認できます。

`append-only`（追記のみ、既存物を書き換えない）と明示しているのも重要です。**監査コードが監査対象を壊さない**ことが保証されています。

In [ ]:
"""Append-only runtime audit for the exact public H&S v38 inference notebook.

This module is embedded verbatim as the final notebook cell.  It must not write
to ``submission.csv``.  Its only prediction-adjacent operation is a byte-for-byte
copy to ``submission_parent_exact.csv`` before validation.
"""

from __future__ import annotations

import hashlib as _i21_hashlib
import json as _i21_json
import os as _i21_os
import platform as _i21_platform
import shutil as _i21_shutil
import traceback as _i21_traceback
from datetime import datetime as _i21_datetime, timezone as _i21_timezone
from pathlib import Path as _I21Path
from typing import Any as _I21Any, Mapping as _I21Mapping

import numpy as _i21_np
import pandas as _i21_pd
import torch as _i21_torch


_I21_LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]
_I21_VISIBLE_UID_SHA256 = (
    "f8308447b47011121885d31b754b690b1bd3b51a78e8f07c0b148161f8396ea0"
)
_I21_VISIBLE_SUBMISSION_SHA256 = (
    "1b03ce7f093efaffdf9acb9e8af05946bdd60b4fec1e0b1fc536b8198abb513b"
)
_I21_PARENT_NOTEBOOK_SHA256 = (
    "af519f52b2473bf7622957c8802da368b5d67cbb899500e8bb944f0b2f2ab0e1"
)
_I21_EXPECTED_RAD_GATE = [
    "ACL",
    "Baker's",
    "Contusion",
    "Effusion",
    "Lateral OA",
    "Medial OA",
    "PF OA",
]
_I21_EXPECTED_RAPTOR_ARMS = [
    {"name": "maxspan-v5", "weight": 0.55, "windows": 62},
    {"name": "native384dense-v10", "weight": 0.10, "windows": 62},
    {"name": "maxspan-v5-reverse", "weight": 0.15, "windows": 62},
    {"name": "native384-v8", "weight": 0.20, "windows": 42},
]


def _i21_sha256_bytes(payload: bytes) -> str:
    return _i21_hashlib.sha256(payload).hexdigest()


def _i21_sha256_file(path: _I21Path) -> str:
    digest = _i21_hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _i21_uid_sha256(uids: list[str]) -> str:
    return _i21_sha256_bytes("".join(f"{uid}\n" for uid in uids).encode("utf-8"))


def _i21_jsonable(value: _I21Any) -> _I21Any:
    if isinstance(value, _I21Path):
        return str(value)
    if isinstance(value, (_i21_np.integer,)):
        return int(value)
    if isinstance(value, (_i21_np.floating,)):
        return float(value)
    if isinstance(value, (_i21_np.bool_,)):
        return bool(value)
    if isinstance(value, tuple):
        return [_i21_jsonable(item) for item in value]
    if isinstance(value, list):
        return [_i21_jsonable(item) for item in value]
    if isinstance(value, dict):
        return {str(key): _i21_jsonable(item) for key, item in value.items()}
    return value


def _i21_write_receipt(path: _I21Path, receipt: dict[str, _I21Any]) -> None:
    path.write_text(
        _i21_json.dumps(
            _i21_jsonable(receipt),
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )


def _i21_find_competition_root(
    namespace: _I21Mapping[str, _I21Any], explicit_root: str | _I21Path | None
) -> _I21Path:
    candidates: list[_I21Path] = []
    if explicit_root is not None:
        candidates.append(_I21Path(explicit_root))
    root_from_parent = namespace.get("ROOT")
    if root_from_parent is not None:
        candidates.append(_I21Path(str(root_from_parent)))
    candidates.extend(
        [
            _I21Path("/kaggle/input/rsna-knee-abnormality-detection"),
            _I21Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        ]
    )
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (candidate / "test.csv").is_file() and (
            candidate / "test_series.csv"
        ).is_file():
            return candidate
    for directory, _, names in _i21_os.walk("/kaggle/input"):
        if "test.csv" in names and "test_series.csv" in names:
            return _I21Path(directory)
    raise RuntimeError("infra0021: competition test.csv/test_series.csv not found")


def _i21_assert_frame(
    frame: _i21_pd.DataFrame,
    expected_uids: list[str],
    *,
    context: str,
) -> _i21_np.ndarray:
    expected_columns = ["StudyInstanceUID", *_I21_LABELS]
    if frame.columns.tolist() != expected_columns:
        raise RuntimeError(
            f"infra0021: {context} schema drift: {frame.columns.tolist()}"
        )
    uids = frame["StudyInstanceUID"].astype(str).tolist()
    if uids != expected_uids:
        raise RuntimeError(f"infra0021: {context} UID/order drift")
    if len(uids) != len(set(uids)):
        raise RuntimeError(f"infra0021: {context} contains duplicate UIDs")
    values = frame[_I21_LABELS].to_numpy(_i21_np.float64)
    if values.shape != (len(expected_uids), len(_I21_LABELS)):
        raise RuntimeError(f"infra0021: {context} prediction shape drift")
    if not _i21_np.isfinite(values).all():
        raise RuntimeError(f"infra0021: {context} contains non-finite predictions")
    if values.size and (float(values.min()) < 0.0 or float(values.max()) > 1.0):
        raise RuntimeError(f"infra0021: {context} predictions outside [0, 1]")
    return values


def _i21_manifest_member_count(namespace: _I21Mapping[str, _I21Any]) -> tuple[int, str]:
    asset = namespace.get("ASSET")
    if asset is None:
        raise RuntimeError("infra0021: parent ASSET variable missing")
    manifest_path = _I21Path(str(asset)) / "rsna-knee-weights" / "manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError(f"infra0021: DINO manifest missing: {manifest_path}")
    manifest = _i21_json.loads(manifest_path.read_text(encoding="utf-8"))
    members = manifest.get("members")
    if not isinstance(members, list) or len(members) != 20:
        raise RuntimeError(
            f"infra0021: expected 20 DINO members, got "
            f"{len(members) if isinstance(members, list) else type(members).__name__}"
        )
    member_ids = [str(member.get("id", "")) for member in members]
    if any(not member_id for member_id in member_ids) or len(member_ids) != len(
        set(member_ids)
    ):
        raise RuntimeError("infra0021: DINO member IDs missing or duplicated")
    return len(members), _i21_sha256_file(manifest_path)


def _i21_validate_a5(
    namespace: _I21Mapping[str, _I21Any], study_count: int
) -> dict[str, _I21Any]:
    models = namespace.get("models")
    preds = namespace.get("preds")
    ok = namespace.get("_a5_ok")
    if not isinstance(models, list) or len(models) != 5:
        raise RuntimeError(
            f"infra0021: expected five A5 models, got "
            f"{len(models) if isinstance(models, list) else type(models).__name__}"
        )
    predictions = _i21_np.asarray(preds)
    expected_shape = (5, study_count, len(_I21_LABELS))
    if predictions.shape != expected_shape:
        raise RuntimeError(
            f"infra0021: A5 prediction shape {predictions.shape}, expected {expected_shape}"
        )
    if not _i21_np.isfinite(predictions).all():
        raise RuntimeError("infra0021: A5 predictions contain fallback NaN/inf")
    valid = _i21_np.asarray(ok, dtype=bool)
    if valid.shape != (study_count,) or not valid.all():
        raise RuntimeError("infra0021: A5 did not complete every study")
    return {
        "model_count": len(models),
        "prediction_shape": list(predictions.shape),
        "all_studies_complete": bool(valid.all()),
    }


def _i21_validate_rad(namespace: _I21Mapping[str, _I21Any]) -> dict[str, _I21Any]:
    if namespace.get("V18_CALIBRATOR_APPLIED") is not True:
        raise RuntimeError("infra0021: V18 Rad/transformer calibrator was not applied")
    gate = sorted(str(item) for item in namespace.get("V18_CAL_GATE", ()))
    if gate != _I21_EXPECTED_RAD_GATE:
        raise RuntimeError(f"infra0021: V18 calibration gate drift: {gate}")
    return {"calibrator_applied": True, "gate": gate}


def _i21_validate_raptor(
    namespace: _I21Mapping[str, _I21Any], expected_uids: list[str]
) -> dict[str, _I21Any]:
    arms = namespace.get("ARMS")
    if not isinstance(arms, list) or len(arms) != 4:
        raise RuntimeError("infra0021: Raptor four-arm definition missing")
    observed = [
        {
            "name": str(arm.get("name", "")),
            "weight": float(arm.get("w", -1.0)),
            "windows": int(arm.get("k_eval", -1)),
        }
        for arm in arms
    ]
    if observed != _I21_EXPECTED_RAPTOR_ARMS:
        raise RuntimeError(f"infra0021: Raptor arm contract drift: {observed}")
    weights = namespace.get("_coatnet_weight")
    if not isinstance(weights, dict) or set(weights) != set(_I21_LABELS):
        raise RuntimeError("infra0021: Raptor outer-weight map missing or incomplete")
    if any(float(weights[label]) != 0.60 for label in _I21_LABELS):
        raise RuntimeError("infra0021: Raptor outer global weight drift")
    coatnet = namespace.get("_blend_coatnet")
    if not isinstance(coatnet, _i21_pd.DataFrame):
        raise RuntimeError(
            "infra0021: CoAtNet branch output unavailable; transformer fallback is forbidden"
        )
    _i21_assert_frame(coatnet, expected_uids, context="CoAtNet branch")
    blended = namespace.get("_blend_output")
    if not isinstance(blended, _i21_pd.DataFrame):
        raise RuntimeError("infra0021: final Raptor blend object missing")
    values = _i21_assert_frame(blended, expected_uids, context="in-memory final blend")
    return {
        "arm_contract": observed,
        "outer_weight": 0.60,
        "final_shape": list(values.shape),
        "branch_output_present": True,
    }


def run_infra0021_audit(
    namespace: _I21Mapping[str, _I21Any],
    *,
    work_dir: str | _I21Path = "/kaggle/working",
    competition_root: str | _I21Path | None = None,
    enforce_gpu: bool = True,
) -> dict[str, _I21Any]:
    """Validate the completed parent notebook without changing its predictions."""

    work = _I21Path(work_dir)
    work.mkdir(parents=True, exist_ok=True)
    receipt_path = work / "infra0021_runtime_receipt.json"
    receipt: dict[str, _I21Any] = {
        "schema_version": "infra0021_runtime_receipt_v1",
        "infra_id": "infra0021_public0936_exact_clone_audit",
        "status": "started",
        "started_at_utc": _i21_datetime.now(_i21_timezone.utc).isoformat(),
        "prediction_recipe_changed": False,
        "competition_submission_performed": False,
        "parent": {
            "kernel": "prvsiyan/head-and-shoulders-knees-and-toes",
            "version": 38,
            "script_version_id": 344807997,
            "public_score_provenance_only": 0.936,
            "source_sha256": _I21_PARENT_NOTEBOOK_SHA256,
        },
        "gates": {},
    }
    _i21_write_receipt(receipt_path, receipt)

    try:
        root = _i21_find_competition_root(namespace, competition_root)
        test_path = root / "test.csv"
        series_path = root / "test_series.csv"
        test = _i21_pd.read_csv(test_path, dtype={"StudyInstanceUID": str})
        if test.columns.tolist().count("StudyInstanceUID") != 1:
            raise RuntimeError("infra0021: test.csv StudyInstanceUID contract drift")
        test_uids = test["StudyInstanceUID"].astype(str).tolist()
        if not test_uids or any(not uid for uid in test_uids):
            raise RuntimeError("infra0021: test.csv contains empty/no UIDs")
        if len(test_uids) != len(set(test_uids)):
            raise RuntimeError("infra0021: test.csv contains duplicate UIDs")
        uid_sha = _i21_uid_sha256(test_uids)
        visible_reference = uid_sha == _I21_VISIBLE_UID_SHA256

        gpu_count = int(_i21_torch.cuda.device_count())
        gpu_names = [
            str(_i21_torch.cuda.get_device_name(index)) for index in range(gpu_count)
        ]
        if enforce_gpu:
            if not _i21_torch.cuda.is_available():
                raise RuntimeError("infra0021: CUDA is unavailable")
            if gpu_count != 2:
                raise RuntimeError(
                    f"infra0021: expected exactly two GPUs, got {gpu_count}"
                )
            if any("T4" not in name.upper() for name in gpu_names):
                raise RuntimeError(f"infra0021: expected Tesla T4 GPUs, got {gpu_names}")

        submission_path = work / "submission.csv"
        if not submission_path.is_file():
            raise RuntimeError("infra0021: parent did not produce submission.csv")
        submission_sha = _i21_sha256_file(submission_path)
        backup_path = work / "submission_parent_exact.csv"
        _i21_shutil.copyfile(submission_path, backup_path)
        backup_sha = _i21_sha256_file(backup_path)
        if backup_sha != submission_sha:
            raise RuntimeError("infra0021: byte backup of submission.csv changed")

        submission = _i21_pd.read_csv(
            submission_path, dtype={"StudyInstanceUID": str}
        )
        submission_values = _i21_assert_frame(
            submission, test_uids, context="submission.csv"
        )
        if visible_reference and submission_sha != _I21_VISIBLE_SUBMISSION_SHA256:
            raise RuntimeError(
                "infra0021: visible three-study output differs from exact v38 parent"
            )

        dino_count, dino_manifest_sha = _i21_manifest_member_count(namespace)
        a5 = _i21_validate_a5(namespace, len(test_uids))
        rad = _i21_validate_rad(namespace)
        raptor = _i21_validate_raptor(namespace, test_uids)
        in_memory = namespace["_blend_output"][_I21_LABELS].to_numpy(
            _i21_np.float64
        )
        max_abs_csv_vs_memory = float(
            _i21_np.max(_i21_np.abs(submission_values - in_memory))
            if submission_values.size
            else 0.0
        )
        if max_abs_csv_vs_memory > 1e-12:
            raise RuntimeError(
                "infra0021: final CSV differs from the in-memory Raptor blend "
                f"(max abs {max_abs_csv_vs_memory:.3e})"
            )

        input_mounts = sorted(
            str(path)
            for path in _I21Path("/kaggle/input").iterdir()
        ) if _I21Path("/kaggle/input").is_dir() else []
        receipt.update(
            {
                "status": "passed",
                "completed_at_utc": _i21_datetime.now(
                    _i21_timezone.utc
                ).isoformat(),
                "input": {
                    "competition_root": str(root),
                    "test_csv_sha256": _i21_sha256_file(test_path),
                    "test_series_csv_sha256": _i21_sha256_file(series_path),
                    "study_count": len(test_uids),
                    "uid_sha256": uid_sha,
                    "mode": "visible_reference" if visible_reference else "dynamic_hidden",
                    "visible_static_output_sha_gate_applied": visible_reference,
                    "mounts": input_mounts,
                },
                "environment": {
                    "python": _i21_platform.python_version(),
                    "torch": str(_i21_torch.__version__),
                    "numpy": str(_i21_np.__version__),
                    "pandas": str(_i21_pd.__version__),
                    "cuda_available": bool(_i21_torch.cuda.is_available()),
                    "gpu_count": gpu_count,
                    "gpu_names": gpu_names,
                },
                "output": {
                    "submission_path": str(submission_path),
                    "submission_sha256": submission_sha,
                    "backup_path": str(backup_path),
                    "backup_sha256": backup_sha,
                    "shape": [len(test_uids), len(_I21_LABELS) + 1],
                    "max_abs_csv_vs_memory": max_abs_csv_vs_memory,
                },
                "gates": {
                    "gpu_t4x2": (not enforce_gpu)
                    or (
                        gpu_count == 2
                        and all("T4" in name.upper() for name in gpu_names)
                    ),
                    "dino": {
                        "member_count": dino_count,
                        "manifest_sha256": dino_manifest_sha,
                    },
                    "a5": a5,
                    "rad": rad,
                    "raptor": raptor,
                    "submission_schema_uid_range": True,
                    "visible_exact_sha256": (
                        submission_sha == _I21_VISIBLE_SUBMISSION_SHA256
                        if visible_reference
                        else "not_applicable_dynamic_input"
                    ),
                },
            }
        )
        _i21_write_receipt(receipt_path, receipt)
        print(
            "infra0021 audit PASSED | "
            f"mode={receipt['input']['mode']} studies={len(test_uids)} "
            f"submission_sha256={submission_sha}",
            flush=True,
        )
        return receipt
    except Exception as exc:
        receipt.update(
            {
                "status": "failed",
                "completed_at_utc": _i21_datetime.now(
                    _i21_timezone.utc
                ).isoformat(),
                "error": {
                    "type": type(exc).__name__,
                    "message": str(exc),
                    "traceback": _i21_traceback.format_exc(),
                },
            }
        )
        _i21_write_receipt(receipt_path, receipt)
        raise


if __name__ == "__main__":
    run_infra0021_audit(globals())


## セル8：内側半月板だけを差し替える最終オーバーレイ

**何をしているか**：このnotebookの**唯一の実質的な変更点**です。

```
final_rank(Medial Meniscus) = 0.30 × Transformer rank
                            + 0.60 × Raptor rank
                            + 0.10 × fullfit0033 bag rank
```

を計算し、親と同じ**平均タイ順位のパーセンタイル再ランク**を掛けて書き戻します。**外側半月板を含む残り11所見は、親の提出物からバイト単位でそのままコピー**されます。

そして書き込む前に、親の提出物とUID一覧のSHA256を再度照合しています。

**なぜそうするのか**：

- **なぜ内側半月板だけか。** macro-AUCは12所見の独立な平均なので、1所見の改善はそのまま `1/12` の重みで総合点に乗り、他所見には一切影響しません。逆に言えば、**1所見に閉じた変更なら、その効果を他の要因と混ぜずに測れます**。もし12所見全部を同時にいじっていたら、+0.001 が「どの所見のどの変更で得られたのか」は永遠に分かりません。**変更を最小単位に閉じ込めることが、そのまま測定可能性になる**——これが本notebookの最大の教訓です。
- **なぜ Raptor の重みが 0.60 と一番高いか。** Raptorは断面方向（Sagittal/Coronal/Axial）ごとにスロットを分けたモデルでした。半月板は**冠状断で内側・外側の区別が最も明瞭**なので、断面を意識したモデルが半月板で強いのは解剖学的に筋が通ります。重みの根拠が「LBで試したら良かった」だけでなく**臨床的に説明できる**のは良い兆候です。
- **なぜランク空間で混ぜるか。** 3つの成分はそれぞれ確率のスケールが違います。順位に変換すれば、スケール差を完全に無視して混ぜられます。AUCは順位しか見ないので、これで失うものはありません。
- **なぜ平均タイ順位（average tie rank）か。** 同じ予測値が複数あるとき、順位の付け方（最小・最大・平均）で結果が変わります。親と同じ方式を使わないと、**変更していないはずの部分の順位まで動いてしまいます**。細部まで親に揃えているのは、比較可能性を守るためです。

**逆算してみると**：総合スコアは 0.936 → 0.937 で +0.001。これは1所見の改善が 1/12 に希釈された結果なので、**内側半月板単体では約 +0.012 の改善**があったことになります。1所見で0.012はかなり大きな改善です。

**残る疑問**：外側半月板（Lateral Meniscus）には同じ処方を当てていません。docstringにも理由は書かれていません。「試したが効かなかった」のか「まだ試していない」のか——ここは改善余地です（READMEの改善提案を参照）。

In [ ]:
"""Notebook cell source: Medial Meniscus T30 / R60 / bag10 overlay.

The exact 0.936 parent remains authoritative for every target except
Medial Meniscus.  For Medial Meniscus only, rebuild the final rank from the
already-computed Transformer and Raptor rank components plus the fullfit0033
bag rank:

    0.30 * Transformer rank
  + 0.60 * Raptor rank
  + 0.10 * fullfit0033 bag rank

and then apply the same final average-tie percentile rerank used by the parent.

Lateral Meniscus and the other ten targets are copied byte-for-byte from the
exact 0.936 parent submission.
"""

import csv as _p33_csv
import hashlib as _p33_hashlib
import json as _p33_json
import math as _p33_math
import os as _p33_os
from pathlib import Path as _P33Path

import numpy as _p33_np


_P33_LABELS = (
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
)

_P33_MENISCUS = ("Medial Meniscus", "Lateral Meniscus")
_P33_CUSTOM_TARGET = "Medial Meniscus"

_P33_HEADER = ("StudyInstanceUID", *_P33_LABELS)
_P33_BAG_HEADER = ("StudyInstanceUID", *_P33_MENISCUS)

_P33_VISIBLE_UID_SHA256 = (
    "f8308447b47011121885d31b754b690b1bd3b51a78e8f07c0b148161f8396ea0"
)
_P33_VISIBLE_PARENT_SHA256 = (
    "1b03ce7f093efaffdf9acb9e8af05946bdd60b4fec1e0b1fc536b8198abb513b"
)

# Previous post-parent residual is disabled.
_P33_PARENT_WEIGHT = _p33_np.float64(1.00)
_P33_BAG_WEIGHT = _p33_np.float64(0.00)

# New Medial-Meniscus-only three-way route.
_P33_T_WEIGHT = _p33_np.float64(0.30)
_P33_R_WEIGHT = _p33_np.float64(0.60)
_P33_CUSTOM_BAG_WEIGHT = _p33_np.float64(0.10)


def _p33_sha256_file(_p33_path):
    _p33_digest = _p33_hashlib.sha256()
    with _p33_path.open("rb") as _p33_handle:
        for _p33_block in iter(lambda: _p33_handle.read(8 << 20), b""):
            _p33_digest.update(_p33_block)
    return _p33_digest.hexdigest()


def _p33_uid_sha256(_p33_uids):
    return _p33_hashlib.sha256(
        "".join(f"{_p33_uid}\n" for _p33_uid in _p33_uids).encode("utf-8")
    ).hexdigest()


def _p33_read_tokens(_p33_path, _p33_expected_header, _p33_context):
    with _p33_path.open("r", encoding="utf-8", newline="") as _p33_handle:
        _p33_rows = list(_p33_csv.reader(_p33_handle))

    if not _p33_rows or tuple(_p33_rows[0]) != tuple(_p33_expected_header):
        raise RuntimeError(
            f"public0033: {_p33_context} schema drift: "
            f"{_p33_rows[0] if _p33_rows else None!r}"
        )

    _p33_data = _p33_rows[1:]
    if not _p33_data or any(
        len(_p33_row) != len(_p33_expected_header)
        for _p33_row in _p33_data
    ):
        raise RuntimeError(
            f"public0033: {_p33_context} row width/emptiness drift"
        )
    return _p33_data


def _p33_matrix(_p33_rows, _p33_columns, _p33_context):
    _p33_values = _p33_np.empty(
        (len(_p33_rows), len(_p33_columns)),
        dtype=_p33_np.float64,
    )

    for _p33_row_index, _p33_row in enumerate(_p33_rows):
        for _p33_column_index, _p33_column in enumerate(_p33_columns):
            try:
                _p33_value = float(_p33_row[_p33_column])
            except (TypeError, ValueError) as _p33_exc:
                raise RuntimeError(
                    f"public0033: {_p33_context} non-numeric value at "
                    f"row={_p33_row_index}, column={_p33_column}"
                ) from _p33_exc

            if (
                not _p33_math.isfinite(_p33_value)
                or _p33_value < 0.0
                or _p33_value > 1.0
            ):
                raise RuntimeError(
                    f"public0033: {_p33_context} value outside [0,1] at "
                    f"row={_p33_row_index}, column={_p33_column}"
                )

            _p33_values[
                _p33_row_index,
                _p33_column_index,
            ] = _p33_value

    return _p33_values


def _p33_average_tie_rank_pct(_p33_values):
    """Pandas rank(method='average', pct=True) over the full test set."""

    _p33_values = _p33_np.asarray(
        _p33_values,
        dtype=_p33_np.float64,
    )

    if _p33_values.ndim != 1 or _p33_values.size == 0:
        raise RuntimeError(
            "public0033: rank input must be a non-empty vector"
        )

    if not _p33_np.isfinite(_p33_values).all():
        raise RuntimeError(
            "public0033: rank input contains non-finite values"
        )

    _p33_order = _p33_np.argsort(
        _p33_values,
        kind="mergesort",
    )
    _p33_sorted = _p33_values[_p33_order]
    _p33_rank = _p33_np.empty(
        _p33_values.size,
        dtype=_p33_np.float64,
    )

    _p33_start = 0
    while _p33_start < _p33_values.size:
        _p33_end = _p33_start + 1

        while (
            _p33_end < _p33_values.size
            and _p33_sorted[_p33_end]
            == _p33_sorted[_p33_start]
        ):
            _p33_end += 1

        _p33_rank[
            _p33_order[_p33_start:_p33_end]
        ] = (
            (_p33_start + 1 + _p33_end)
            / 2.0
            / _p33_values.size
        )

        _p33_start = _p33_end

    return _p33_rank


def _p33_token_digest(_p33_rows, _p33_column):
    _p33_digest = _p33_hashlib.sha256()

    for _p33_row in _p33_rows:
        _p33_digest.update(
            _p33_row[0].encode("utf-8")
        )
        _p33_digest.update(b"\x1f")
        _p33_digest.update(
            _p33_row[_p33_column].encode("utf-8")
        )
        _p33_digest.update(b"\n")

    return _p33_digest.hexdigest()


def _p33_competition_root():
    _p33_candidates = []

    _p33_explicit = _p33_os.environ.get(
        "PUBLIC0033_COMPETITION_ROOT"
    )
    if _p33_explicit:
        _p33_candidates.append(
            _P33Path(_p33_explicit)
        )

    _p33_parent_root = globals().get("ROOT")
    if _p33_parent_root is not None:
        _p33_candidates.append(
            _P33Path(str(_p33_parent_root))
        )

    _p33_candidates.extend(
        [
            _P33Path(
                "/kaggle/input/rsna-knee-abnormality-detection"
            ),
            _P33Path(
                "/kaggle/input/competitions/rsna-knee-abnormality-detection"
            ),
        ]
    )

    _p33_seen = set()

    for _p33_candidate in _p33_candidates:
        _p33_key = str(_p33_candidate)

        if _p33_key in _p33_seen:
            continue

        _p33_seen.add(_p33_key)

        if (_p33_candidate / "test.csv").is_file():
            return _p33_candidate

    raise RuntimeError(
        "public0033: competition test.csv is unavailable"
    )


def _p33_test_uids(_p33_root):
    with (
        _p33_root / "test.csv"
    ).open(
        "r",
        encoding="utf-8",
        newline="",
    ) as _p33_handle:
        _p33_rows = list(
            _p33_csv.reader(_p33_handle)
        )

    if (
        not _p33_rows
        or _p33_rows[0].count(
            "StudyInstanceUID"
        ) != 1
    ):
        raise RuntimeError(
            "public0033: test.csv StudyInstanceUID schema drift"
        )

    _p33_index = _p33_rows[0].index(
        "StudyInstanceUID"
    )

    _p33_uids = [
        _p33_row[_p33_index]
        for _p33_row in _p33_rows[1:]
    ]

    if (
        not _p33_uids
        or any(
            not _p33_uid
            for _p33_uid in _p33_uids
        )
    ):
        raise RuntimeError(
            "public0033: test.csv contains empty UIDs"
        )

    if len(set(_p33_uids)) != len(_p33_uids):
        raise RuntimeError(
            "public0033: test.csv contains duplicate UIDs"
        )

    return _p33_uids


def _p33_main():
    _p33_work = _P33Path(
        _p33_os.environ.get(
            "PUBLIC0033_WORK_DIR",
            "/kaggle/working",
        )
    )

    _p33_parent_path = (
        _p33_work
        / "submission_parent_exact.csv"
    )
    _p33_current_path = (
        _p33_work
        / "submission.csv"
    )
    _p33_bag_path = (
        _p33_work
        / "public0033_bag_raw.csv"
    )
    _p33_audit_path = (
        _p33_work
        / "infra0021_runtime_receipt.json"
    )
    _p33_receipt_path = (
        _p33_work
        / "public0033_overlay_receipt.json"
    )
    _p33_temp_path = (
        _p33_work
        / "submission_public0033.tmp.csv"
    )

    for _p33_path, _p33_name in (
        (
            _p33_parent_path,
            "parent backup",
        ),
        (
            _p33_current_path,
            "current parent submission",
        ),
        (
            _p33_bag_path,
            "bag raw prediction",
        ),
        (
            _p33_audit_path,
            "infra0021 audit receipt",
        ),
    ):
        if not _p33_path.is_file():
            raise RuntimeError(
                f"public0033: required "
                f"{_p33_name} is missing: "
                f"{_p33_path}"
            )

    # The custom route requires the two components
    # created by the upstream Raptor blending cell.
    if (
        "_blend_tr" not in globals()
        or "_blend_cr" not in globals()
        or "_blend_transformer" not in globals()
        or "_blend_coatnet" not in globals()
    ):
        raise RuntimeError(
            "public0033: Transformer/Raptor blend "
            "components are unavailable"
        )

    if _P33_CUSTOM_TARGET not in _blend_tr.columns:
        raise RuntimeError(
            "public0033: Medial Meniscus missing "
            "from Transformer rank component"
        )

    if _P33_CUSTOM_TARGET not in _blend_cr.columns:
        raise RuntimeError(
            "public0033: Medial Meniscus missing "
            "from Raptor rank component"
        )

    _p33_audit = _p33_json.loads(
        _p33_audit_path.read_text(
            encoding="utf-8"
        )
    )

    if _p33_audit.get("status") != "passed":
        raise RuntimeError(
            "public0033: infra0021 audit did not pass"
        )

    _p33_parent_sha = _p33_sha256_file(
        _p33_parent_path
    )

    if (
        _p33_sha256_file(_p33_current_path)
        != _p33_parent_sha
    ):
        raise RuntimeError(
            "public0033: submission.csv changed "
            "after the parent audit"
        )

    _p33_parent_rows = _p33_read_tokens(
        _p33_parent_path,
        _P33_HEADER,
        "parent backup",
    )

    _p33_parent_uids = [
        _p33_row[0]
        for _p33_row in _p33_parent_rows
    ]

    if (
        any(
            not _p33_uid
            for _p33_uid in _p33_parent_uids
        )
        or len(
            set(_p33_parent_uids)
        )
        != len(_p33_parent_uids)
    ):
        raise RuntimeError(
            "public0033: parent backup UID identity drift"
        )

    _p33_root = _p33_competition_root()
    _p33_test = _p33_test_uids(
        _p33_root
    )

    if _p33_parent_uids != _p33_test:
        raise RuntimeError(
            "public0033: parent backup UID order "
            "differs from test.csv"
        )

    # Explicitly verify that upstream rank components
    # have the same row identity as the parent.
    _p33_transformer_uids = (
        _blend_transformer[
            "StudyInstanceUID"
        ]
        .astype(str)
        .tolist()
    )

    _p33_raptor_uids = (
        _blend_coatnet[
            "StudyInstanceUID"
        ]
        .astype(str)
        .tolist()
    )

    if (
        _p33_transformer_uids
        != _p33_parent_uids
    ):
        raise RuntimeError(
            "public0033: Transformer component UID order drift"
        )

    if (
        _p33_raptor_uids
        != _p33_parent_uids
    ):
        raise RuntimeError(
            "public0033: Raptor component UID order drift"
        )

    _p33_uid_sha = _p33_uid_sha256(
        _p33_parent_uids
    )

    _p33_visible_reference = (
        _p33_uid_sha
        == _P33_VISIBLE_UID_SHA256
    )

    if (
        _p33_visible_reference
        and _p33_parent_sha
        != _P33_VISIBLE_PARENT_SHA256
    ):
        raise RuntimeError(
            "public0033: visible parent "
            "submission SHA gate failed"
        )

    _p33_parent_values = _p33_matrix(
        _p33_parent_rows,
        tuple(
            range(
                1,
                len(_P33_HEADER),
            )
        ),
        "parent backup",
    )

    _p33_bag_rows = _p33_read_tokens(
        _p33_bag_path,
        _P33_BAG_HEADER,
        "bag raw",
    )

    _p33_bag_by_uid = {}

    for _p33_row in _p33_bag_rows:
        _p33_uid = _p33_row[0]

        if (
            not _p33_uid
            or _p33_uid
            in _p33_bag_by_uid
        ):
            raise RuntimeError(
                "public0033: bag raw UID identity drift"
            )

        _p33_bag_by_uid[
            _p33_uid
        ] = _p33_row

    if (
        set(_p33_bag_by_uid)
        != set(_p33_parent_uids)
    ):
        raise RuntimeError(
            "public0033: bag raw UID set "
            "differs from parent backup"
        )

    _p33_bag_rows_ordered = [
        _p33_bag_by_uid[_p33_uid]
        for _p33_uid in _p33_parent_uids
    ]

    _p33_bag_values = _p33_matrix(
        _p33_bag_rows_ordered,
        (1, 2),
        "bag raw",
    )

    _p33_final_rows = [
        list(_p33_row)
        for _p33_row in _p33_parent_rows
    ]

    _p33_final_values = (
        _p33_parent_values.copy()
    )

    # All eleven untouched targets must remain byte-identical
    # to the exact 0.936 parent.
    _p33_untouched_columns = []
    _p33_untouched_digests_parent = {}

    for (
        _p33_target_index,
        _p33_target,
    ) in enumerate(_P33_LABELS):

        _p33_column = (
            _p33_target_index + 1
        )

        if (
            _p33_target
            != _P33_CUSTOM_TARGET
        ):
            _p33_untouched_columns.append(
                _p33_column
            )

            _p33_untouched_digests_parent[
                _p33_target
            ] = _p33_token_digest(
                _p33_parent_rows,
                _p33_column,
            )

            continue

        # Transformer rank and Raptor rank are exactly the
        # pre-final-rerank components used by the parent.
        _p33_tr_rank = _p33_np.asarray(
            _blend_tr[
                _P33_CUSTOM_TARGET
            ].to_numpy(),
            dtype=_p33_np.float64,
        )

        _p33_r_rank = _p33_np.asarray(
            _blend_cr[
                _P33_CUSTOM_TARGET
            ].to_numpy(),
            dtype=_p33_np.float64,
        )

        _p33_bag_rank = (
            _p33_average_tie_rank_pct(
                _p33_bag_values[
                    :,
                    _P33_MENISCUS.index(
                        _P33_CUSTOM_TARGET
                    ),
                ]
            )
        )

        if (
            _p33_tr_rank.shape
            != _p33_bag_rank.shape
            or _p33_r_rank.shape
            != _p33_bag_rank.shape
        ):
            raise RuntimeError(
                "public0033: custom component "
                "shape mismatch"
            )

        if not (
            _p33_np.isfinite(
                _p33_tr_rank
            ).all()
            and _p33_np.isfinite(
                _p33_r_rank
            ).all()
            and _p33_np.isfinite(
                _p33_bag_rank
            ).all()
        ):
            raise RuntimeError(
                "public0033: custom component "
                "contains non-finite values"
            )

        # Directly replace a portion of the Transformer vote.
        # Raptor remains at the parent's original 60%.
        _p33_pre_rank = (
            _P33_T_WEIGHT
            * _p33_tr_rank
            + _P33_R_WEIGHT
            * _p33_r_rank
            + _P33_CUSTOM_BAG_WEIGHT
            * _p33_bag_rank
        )

        if (
            not _p33_np.isfinite(
                _p33_pre_rank
            ).all()
            or _p33_pre_rank.min()
            < 0.0
            or _p33_pre_rank.max()
            > 1.0
        ):
            raise RuntimeError(
                "public0033: invalid pre-rerank "
                "Medial Meniscus values"
            )

        # Match the parent's final Raptor-blend contract.
        _p33_blended = (
            _p33_average_tie_rank_pct(
                _p33_pre_rank
            )
        )

        if (
            not _p33_np.isfinite(
                _p33_blended
            ).all()
            or _p33_blended.min()
            < 0.0
            or _p33_blended.max()
            > 1.0
        ):
            raise RuntimeError(
                "public0033: invalid final "
                "Medial Meniscus values"
            )

        _p33_final_values[
            :,
            _p33_target_index,
        ] = _p33_blended

        for (
            _p33_row_index,
            _p33_value,
        ) in enumerate(
            _p33_blended
        ):
            _p33_final_rows[
                _p33_row_index
            ][
                _p33_column
            ] = format(
                float(_p33_value),
                ".17g",
            )

    if any(
        _p33_final_row[0]
        != _p33_parent_row[0]
        for (
            _p33_final_row,
            _p33_parent_row,
        ) in zip(
            _p33_final_rows,
            _p33_parent_rows,
        )
    ):
        raise RuntimeError(
            "public0033: output UID token changed"
        )

    _p33_untouched_digests_final = {}

    for (
        _p33_target_index,
        _p33_target,
    ) in enumerate(_P33_LABELS):

        if (
            _p33_target
            == _P33_CUSTOM_TARGET
        ):
            continue

        _p33_column = (
            _p33_target_index + 1
        )

        if any(
            _p33_final_row[
                _p33_column
            ]
            != _p33_parent_row[
                _p33_column
            ]
            for (
                _p33_final_row,
                _p33_parent_row,
            ) in zip(
                _p33_final_rows,
                _p33_parent_rows,
            )
        ):
            raise RuntimeError(
                "public0033: untouched raw "
                f"token changed: {_p33_target}"
            )

        _p33_untouched_digests_final[
            _p33_target
        ] = _p33_token_digest(
            _p33_final_rows,
            _p33_column,
        )

    if (
        _p33_untouched_digests_final
        != _p33_untouched_digests_parent
    ):
        raise RuntimeError(
            "public0033: untouched raw-token "
            "digest mismatch"
        )

    if _p33_temp_path.exists():
        _p33_temp_path.unlink()

    with _p33_temp_path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as _p33_handle:

        _p33_writer = _p33_csv.writer(
            _p33_handle,
            lineterminator="\n",
        )

        _p33_writer.writerow(
            _P33_HEADER
        )

        _p33_writer.writerows(
            _p33_final_rows
        )

    _p33_reloaded_rows = (
        _p33_read_tokens(
            _p33_temp_path,
            _P33_HEADER,
            "temporary overlay",
        )
    )

    _p33_reloaded_uids = [
        _p33_row[0]
        for _p33_row
        in _p33_reloaded_rows
    ]

    if (
        _p33_reloaded_uids
        != _p33_parent_uids
    ):
        raise RuntimeError(
            "public0033: temporary overlay "
            "UID order drift"
        )

    _p33_reloaded_values = (
        _p33_matrix(
            _p33_reloaded_rows,
            tuple(
                range(
                    1,
                    len(_P33_HEADER),
                )
            ),
            "temporary overlay",
        )
    )

    if not _p33_np.array_equal(
        _p33_reloaded_values,
        _p33_final_values,
    ):
        _p33_max_abs = float(
            _p33_np.max(
                _p33_np.abs(
                    _p33_reloaded_values
                    - _p33_final_values
                )
            )
        )

        raise RuntimeError(
            "public0033: CSV/in-memory gate "
            f"failed (max abs "
            f"{_p33_max_abs:.3e})"
        )

    for (
        _p33_target_index,
        _p33_target,
    ) in enumerate(_P33_LABELS):

        if (
            _p33_target
            == _P33_CUSTOM_TARGET
        ):
            continue

        _p33_column = (
            _p33_target_index + 1
        )

        if (
            _p33_token_digest(
                _p33_reloaded_rows,
                _p33_column,
            )
            != _p33_untouched_digests_parent[
                _p33_target
            ]
        ):
            raise RuntimeError(
                "public0033: temporary untouched "
                f"token drift: {_p33_target}"
            )

    _p33_os.replace(
        _p33_temp_path,
        _p33_current_path,
    )

    _p33_output_sha = (
        _p33_sha256_file(
            _p33_current_path
        )
    )

    _p33_receipt = {
        "schema_version": (
            "public0033_medial_t30_r60_b10_overlay_v1"
        ),
        "status": "passed",
        "parent_authority": str(
            _p33_parent_path
        ),
        "parent_submission_sha256": (
            _p33_parent_sha
        ),
        "parent_uid_sha256": (
            _p33_uid_sha
        ),
        "visible_parent_sha_gate_applied": (
            _p33_visible_reference
        ),
        "legacy_post_parent_residual": {
            "parent_rank_weight": float(
                _P33_PARENT_WEIGHT
            ),
            "bag_rank_weight": float(
                _P33_BAG_WEIGHT
            ),
            "enabled": False,
        },
        "bag_raw_path": str(
            _p33_bag_path
        ),
        "bag_raw_sha256": (
            _p33_sha256_file(
                _p33_bag_path
            )
        ),
        "targets_changed": [
            _P33_CUSTOM_TARGET
        ],
        "custom_route": {
            "target": (
                _P33_CUSTOM_TARGET
            ),
            "transformer_rank_weight": float(
                _P33_T_WEIGHT
            ),
            "raptor_rank_weight": float(
                _P33_R_WEIGHT
            ),
            "bag_rank_weight": float(
                _P33_CUSTOM_BAG_WEIGHT
            ),
            "weights_sum": float(
                _P33_T_WEIGHT
                + _P33_R_WEIGHT
                + _P33_CUSTOM_BAG_WEIGHT
            ),
            "final_rerank": True,
        },
        "rank": (
            "float64_average_tie_percentile_rank_over_all_test_studies"
        ),
        "untouched_targets": [
            _p33_target
            for _p33_target
            in _P33_LABELS
            if _p33_target
            != _P33_CUSTOM_TARGET
        ],
        "untouched_target_raw_token_sha256": (
            _p33_untouched_digests_final
        ),
        "output_path": str(
            _p33_current_path
        ),
        "output_sha256": (
            _p33_output_sha
        ),
        "csv_vs_memory_max_abs": 0.0,
    }

    _p33_receipt_path.write_text(
        _p33_json.dumps(
            _p33_receipt,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    print(
        "public0033 Medial T30/R60/bag10 "
        "overlay PASSED | "
        f"studies={len(_p33_parent_uids)} "
        f"output_sha256={_p33_output_sha}",
        flush=True,
    )


_p33_main()